<p class="lead">This supplement has no lecture of its own. It exists because every unit from Unit 3 onwards reports a number — an accuracy, an area under a curve, an error — and the official plan never says where such numbers come from or when they can be believed. The slides of those units are a map; this text is the territory beneath the number. Read it linearly once, executing every chunk, then use it as a reference. It answers four questions: how to estimate performance on patients the model has never seen (Sections 1–4); which number to report and what it means clinically (Sections 5–6); how evaluations go wrong without anyone noticing (Section 7); and what must be reported so that others can judge the model (Section 8). Section 9 packages the answers into one function that the later units reuse. This is the R edition; only the language and libraries differ from the Python version, the ideas and the references are identical.</p>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Explain why the error measured on the training sample is not an estimate of the error on new patients, and define the roles of the training, validation and test samples.</li>
<li>Implement k-fold cross-validation by hand in base R (index arithmetic), describe the stratified, repeated, grouped and leave-one-out variants, and explain why hyperparameter selection requires nested cross-validation.</li>
<li>Describe overfitting in terms of the bias–variance decomposition and read a complexity curve and a learning curve.</li>
<li>Compute, from a confusion matrix, sensitivity, specificity, precision (PPV), NPV, F1, balanced accuracy and the Matthews correlation coefficient; explain why accuracy fails under class imbalance and why PPV depends on prevalence.</li>
<li>Construct a ROC curve and a precision–recall curve from scores, compute the AUC by hand and with <code>pROC</code>, choose a decision threshold from the costs of the two errors, and assess calibration with a reliability diagram and the Brier score.</li>
<li>Compute MSE, RMSE, MAE and \(R^2\) for a regression model and explain why \(R^2\) can be negative on a test set.</li>
<li>Recognise and prevent the three classic forms of data leakage, and use grouped splits when patients contribute several records.</li>
<li>Distinguish internal from external validation and list what the TRIPOD+AI statement requires a report to contain.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Each code chunk is short and commented. Run it, read its output, then read the paragraph that interprets it. Boxes marked <em>Exercise</em> ask you to modify the chunk above and re-run. Green boxes carry key ideas, orange boxes common misconceptions, purple boxes links to the units that use this material. Only base R and the packages <code>class</code>, <code>e1071</code>, <code>MASS</code> and <code>pROC</code> are used, with <code>mlbench</code> for the data; in a fresh Colab R runtime run <code>install.packages(c("e1071","mlbench","pROC"))</code> once, and choose <em>Runtime → Change runtime type → R</em>. Nothing is downloaded. The recurring dataset is the Wisconsin Diagnostic Breast Cancer sample of Unit 1 (683 complete aspirates, nine nuclear-feature scores, biopsy label),<sup class="c"><a href="#r20">[20]</a></sup> supplied by the <code>mlbench</code> package;<sup class="c"><a href="#r35">[35]</a></sup> regression examples use the Boston housing data,<sup class="c"><a href="#r36">[36]</a></sup> the R stand-in for the diabetes-progression data of Efron and colleagues used in the Python edition.<sup class="c"><a href="#r21">[21]</a></sup></p></div>



In [ ]:
# Setup: libraries used throughout the notebook. The seed is fixed so that every
# number quoted in the text is reproducible.
suppressPackageStartupMessages({
  library(class); library(e1071); library(MASS); library(pROC); library(mlbench)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#8a8a8a"
cat("R", as.character(getRversion()),
    "| class", as.character(packageVersion("class")),
    "| e1071", as.character(packageVersion("e1071")),
    "| pROC", as.character(packageVersion("pROC")),
    "| mlbench", as.character(packageVersion("mlbench")), "\n")



The printout records the versions these notes were built with. One seed at the top is the single most important reproducibility habit (Supplement S0, Section 7).

## 1. Why evaluation is the hardest part

### 1.1 The generalisation question

A supervised model is a function \(f\) fitted to a training sample \(\{(\mathbf{x}_i, y_i)\}_{i=1}^{n}\) drawn from a population of patients, and used on patients who are not in the sample. The quantity we care about is the <strong>expected loss on a new case</strong>,

<div class="eq">\[ \mathrm{Err} = \mathbb{E}_{(\mathbf{x}, y) \sim P}\big[L\big(y, f(\mathbf{x})\big)\big], \]
<span class="where">where \(P\) is the population distribution of patients, \(L\) is a loss function (zero–one loss for classification, squared error for regression) and the expectation is over patients the model has never seen. Hastie, Tibshirani and Friedman call this the <em>test error</em> or <em>generalisation error</em>.<sup class="c"><a href="#r18">[18]</a></sup></span></div>

We cannot compute \(\mathrm{Err}\) because we have a sample, not \(P\). Evaluation methodology is the problem of estimating \(\mathrm{Err}\) from the sample without fooling ourselves, and it is harder than fitting the model, because the model was chosen to make the sample look good.

### 1.2 Training error is not a measurement

The obvious estimate is the error on the training sample itself, the <strong>training error</strong> (or <em>resubstitution error</em>). It is biased downwards, increasingly so with the flexibility of the model, because the same data were used twice: to choose \(f\) and to judge it. The extreme case needs no statistics: the 1-nearest-neighbour classifier (Unit 3) predicts for a case the label of the closest training case; on the training sample the closest case is the case itself, so training accuracy is essentially 100 % by construction, whatever the data.



In [ ]:
# Training accuracy versus accuracy on held-out cases, for the 1-nearest-neighbour classifier.
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                   # drop the Id column
bc <- bc[complete.cases(bc), ]                             # drop the 16 rows with missing values
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))  # ordinal factor -> numeric
X <- as.matrix(bc[, 1:9])                                  # nine nuclear-feature scores (1-10)
feature_names <- colnames(X)
y_pos <- as.integer(bc$Class == "malignant")               # 1 = malignant (condition to detect), 0 = benign
Xs <- scale(X)                                             # standardised features, reused below

set.seed(0)
idx <- sample(nrow(X), round(0.7 * nrow(X)))               # 70% train / 30% test
ctr <- scale(X[idx, ])
cte <- scale(X[-idx, ], center = attr(ctr, "scaled:center"), scale = attr(ctr, "scaled:scale"))
tr_pred <- knn(ctr, ctr, factor(y_pos[idx]), k = 1)
te_pred <- knn(ctr, cte, factor(y_pos[idx]), k = 1)
cat(sprintf("training cases: %d, held-out cases: %d\n", length(idx), nrow(X) - length(idx)))
cat(sprintf("1-NN accuracy on the training sample : %.3f\n", mean(tr_pred == y_pos[idx])))
cat(sprintf("1-NN accuracy on the held-out sample : %.3f\n", mean(te_pred == y_pos[-idx])))



The training accuracy is 1.000 and says nothing; the held-out accuracy, around 0.95, is the first number in this notebook that deserves the name <em>measurement</em>. The gap is the <strong>optimism</strong> of the training error: maximal for 1-NN, small for a linear model with few parameters, potentially total again for a network with millions of parameters, and never zero. (A caveat specific to this dataset: its nine features are integer scores 1–10, so some aspirates share an identical feature vector; <code>knn</code> breaks the resulting distance ties at random, which is why one should not read more than "essentially perfect" into the training figure.)

### 1.3 Three samples, three roles

Practice separates the data into up to three disjoint samples, each with one role.

| Sample | Role | Used how many times? |
|---|---|---|
| **Training** | Fit the parameters of the model (the weights, the tree splits, the stored neighbours) | As often as needed |
| **Validation** (development, tuning) | Choose between models and <em>hyperparameters</em> — the settings the fitting algorithm does not learn, such as \(k\) in k-NN, the depth of a tree, the regularisation strength | As often as needed, but every look costs a little optimism |
| **Test** (hold-out, evaluation) | Estimate the generalisation error of the final model | **Once.** A test set consulted twice has become a validation set |

The usual proportions are 60–20–20, 70–15–15 or 80–10–10 (training–validation–test). There is no universal rule: a larger training share gives a better model, a larger test share a more precise measurement (Section 2), and with very large datasets a 1 % test set already contains more cases than a small study. The third row of the table is the rule most often broken: if the test set is used to compare ten models and the best is reported, the number is the maximum of ten noisy estimates and is biased upwards — the data were again used twice, to choose and to judge (Section 3.4 measures this). When data are scarce, cross-validation inside the training sample plays the validation role (Section 3) and the test set is kept for the end.



In [ ]:
# Schematic of the three samples and their roles.
par(mar = c(0, 0, 0, 0))
plot(NA, xlim = c(0, 100), ylim = c(-0.4, 2.7), axes = FALSE, xlab = "", ylab = "")
blocks <- list(c(0, 60), c(60, 20), c(80, 20)); cols <- c(BLUE, GREEN, ORANGE)
labs <- c("TRAINING\n(fit parameters)", "VALIDATION\n(choose model &\nhyperparameters)", "TEST\n(estimate error,\nused once)")
for (i in 1:3) { x0 <- blocks[[i]][1]; w <- blocks[[i]][2]
  rect(x0, 1, x0 + w, 2, col = adjustcolor(cols[i], 0.85), border = NA)
  text(x0 + w / 2, 1.5, labs[i], col = "white", font = 2, cex = 0.85) }
arrows(0, 0.55, 60, 0.55, code = 3, length = 0.08, col = GREY); text(30, 0.25, "many fits, many looks", cex = 0.85, col = GREY)
arrows(60, 0.55, 80, 0.55, code = 3, length = 0.08, col = GREY); text(70, 0.22, "many looks\n(each costs optimism)", cex = 0.8, col = GREY)
arrows(80, 0.55, 100, 0.55, code = 3, length = 0.08, col = ORANGE); text(90, 0.25, "ONE look", cex = 0.85, col = ORANGE, font = 2)
text(50, 2.4, "all available labelled cases (one random, stratified shuffle)", cex = 0.95)



<p class="caption"><b>Figure 1.</b> The three disjoint samples and their roles. The proportions (here 60/20/20) are conventional, not sacred; what matters is that parameters are never fitted on the validation or test sample, and that the test sample is examined once, after every choice has been made.</p>

<div class="box warn"><h4>Common misconception</h4><p>"Our model reached 99 % accuracy" is not a result until three questions are answered: on which cases (training, validation or test), separated how (random split, by hospital, by date), and how many (a test set of 20 patients cannot distinguish 99 % from 90 %). Every number later in this course should be read with those three questions attached.</p></div>

## 2. Hold-out and its variance

### 2.1 A single split is a single draw

The simplest honest estimate is the <strong>hold-out</strong> method: split at random into a training part and a test part, fit on the first, score on the second. Its weakness is that the estimate depends on which cases landed in the test part. With \(n_{\text{test}}\) test cases and true accuracy \(p\), the number of correct predictions is approximately binomial, so

<div class="eq">\[ \mathrm{SE}(\hat{p}) \approx \sqrt{\frac{\hat{p}(1-\hat{p})}{n_{\text{test}}}}, \]
<span class="where">which for \(\hat{p} = 0.96\) and \(n_{\text{test}} \approx 205\) is about 0.014: two models whose hold-out accuracies differ by one point on 205 cases are not distinguishable. Kohavi's 1995 study is the classical empirical comparison of hold-out, cross-validation and the bootstrap, and the source of the conventional recommendation of stratified ten-fold cross-validation.<sup class="c"><a href="#r2">[2]</a></sup></span></div>

The experiment repeats the 70/30 split 200 times with different seeds, fitting the same 5-nearest-neighbours classifier each time.



In [ ]:
# Repeated hold-out: 200 random 70/30 splits of the same data and the same model (5-NN on standardised features).
accs <- sapply(0:199, function(s) {
  set.seed(s); id <- sample(nrow(X), round(0.7 * nrow(X)))     # no stratification here, on purpose
  tr <- scale(X[id, ]); te <- scale(X[-id, ], center = attr(tr, "scaled:center"), scale = attr(tr, "scaled:scale"))
  mean(knn(tr, te, factor(y_pos[id]), k = 5) == y_pos[-id])
})
n_test <- nrow(X) - round(0.7 * nrow(X))
se_bin <- sqrt(mean(accs) * (1 - mean(accs)) / n_test)
cat(sprintf("200 hold-out accuracies: mean %.3f, SD %.3f, min %.3f, max %.3f\n", mean(accs), sd(accs), min(accs), max(accs)))
cat(sprintf("binomial standard error predicted for n_test = %d: %.3f\n", n_test, se_bin))
hist(accs, breaks = 18, col = BLUE, border = "white", xlab = "test accuracy of the same model on a different random 30% test set",
     ylab = "number of splits", main = "The hold-out estimate is a random variable")
abline(v = mean(accs), col = ORANGE, lwd = 2); abline(v = accs[1], col = GREEN, lwd = 2, lty = 2)
legend("topleft", bty = "n", cex = 0.8, lty = c(1, 2), lwd = 2, col = c(ORANGE, GREEN),
       legend = c(sprintf("mean = %.3f", mean(accs)), sprintf("the split with seed 0: %.3f", accs[1])))



<p class="caption"><b>Figure 2.</b> Distribution of the test accuracy of one fixed model over 200 random hold-out splits. The spread (standard deviation about 0.011) is close to what the binomial formula predicts (about 0.012) for 205 test cases, and the range from the worst to the best split spans several percentage points. The dashed line is the single number that a notebook with <code>set.seed(0)</code> would have reported.</p>

A hold-out accuracy is one draw from a distribution whose width is set by the test size. Reporting it without the test size, or comparing two models on different splits, is reporting noise. Two remedies follow: test on all the data by rotation (cross-validation, Section 3), and report an interval rather than a point.

### 2.2 Stratification

A random split can also, by chance, put different class proportions in the test and training parts, which biases the estimate and increases its variance. <strong>Stratified</strong> splitting draws the split within each class so that proportions are preserved. With 683 cases the effect is small; with the sample sizes of a clinical pilot study it is not, and a class can vanish from the test part altogether. The chunk shrinks the test set to 20 cases to make the effect visible.



In [ ]:
# Stratification: the fraction of malignant cases in a 20-case test set, with and without stratification, over 300 splits.
malignant_fraction <- function(stratified, n_splits = 300, test_size = 20) {
  sapply(0:(n_splits - 1), function(s) {
    set.seed(s)
    if (stratified) {
      n_pos <- round(test_size * mean(y_pos == 1))
      te <- c(sample(which(y_pos == 1), n_pos), sample(which(y_pos == 0), test_size - n_pos))
    } else te <- sample(nrow(X), test_size)
    mean(y_pos[te] == 1)
  })
}
plain <- malignant_fraction(FALSE); strat <- malignant_fraction(TRUE)
tab <- data.frame(min = c(min(plain), min(strat)), mean = c(mean(plain), mean(strat)), max = c(max(plain), max(strat)),
                  SD = c(sd(plain), sd(strat)), `splits with < 20% malignant` = c(sum(plain < 0.2), sum(strat < 0.2)),
                  row.names = c("random split", "stratified split"), check.names = FALSE)
cat(sprintf("fraction of malignant cases in the whole sample: %.3f\n", mean(y_pos == 1)))
round(tab, 3)



Without stratification the malignant fraction of a 20-case test set wanders widely from split to split; with it, every split contains the same number of malignant cases, so the per-class metrics of Section 5 have a stable denominator. Stratification costs nothing and is the default for classification in this course. For regression the analogous precaution is to check that the training and test ranges of the target overlap.

<div class="box"><h4>Exercise 1</h4><p>In the repeated hold-out chunk, change the training fraction from 0.7 to 0.9 and then to 0.5, and watch the standard deviation of the 200 accuracies. A smaller test set makes the <em>estimate</em> noisier; a smaller training set makes the <em>model</em> worse. Which of the two effects dominates at each extreme? The trade-off is the reason cross-validation exists.</p></div>

## 3. Cross-validation

### 3.1 The idea

<strong>k-fold cross-validation</strong> (CV) resolves the trade-off of Exercise 1 by rotation. The sample is partitioned into \(k\) disjoint folds of nearly equal size; for each fold \(j\) the model is fitted on the other \(k-1\) folds and scored on fold \(j\). Every case is tested exactly once, by a model that did not see it. The CV estimate is the average of the \(k\) fold scores,

<div class="eq">\[ \widehat{\mathrm{Err}}_{\text{CV}} = \frac{1}{k} \sum_{j=1}^{k} \mathrm{Err}_j, \qquad \mathrm{Err}_j = \frac{1}{|F_j|} \sum_{i \in F_j} L\big(y_i, f^{(-j)}(\mathbf{x}_i)\big), \]
<span class="where">where \(F_j\) is the set of indices in fold \(j\) and \(f^{(-j)}\) the model fitted without it. Stone formalised the method in 1974 and proved the point Section 3.4 demonstrates: the same data cannot both select a model and assess it.<sup class="c"><a href="#r1">[1]</a></sup> \(k = 5\) or \(10\) is conventional; \(k = n\) is <em>leave-one-out</em> (LOO).</span></div>

Note what CV estimates: each \(f^{(-j)}\) is trained on \((k-1)/k\) of the data, so CV estimates the error of the <em>procedure</em> at roughly that sample size, not of the final model fitted on all the data, which is expected to be slightly better. The CV number is thus mildly conservative — the right direction to err in.

### 3.2 Worked example by hand

Take \(n = 10\) cases with indices 1–10 and \(k = 5\). Without shuffling, consecutive blocks of \(n/k = 2\) cases become each test fold:

| fold \(j\) | test indices \(F_j\) | training indices | fitted model | scored on |
|---|---|---|---|---|
| 1 | {1, 2} | {3, 4, 5, 6, 7, 8, 9, 10} | \(f^{(-1)}\) | cases 1, 2 |
| 2 | {3, 4} | {1, 2, 5, 6, 7, 8, 9, 10} | \(f^{(-2)}\) | cases 3, 4 |
| 3 | {5, 6} | {1, 2, 3, 4, 7, 8, 9, 10} | \(f^{(-3)}\) | cases 5, 6 |
| 4 | {7, 8} | {1, 2, 3, 4, 5, 6, 9, 10} | \(f^{(-4)}\) | cases 7, 8 |
| 5 | {9, 10} | {1, 2, 3, 4, 5, 6, 7, 8} | \(f^{(-5)}\) | cases 9, 10 |

Suppose the fold accuracies are 1.0, 0.5, 1.0, 1.0 and 0.5. The CV accuracy is their mean, \(0.80\), which here equals the pooled \(8/10\) because the folds are equal in size. If the cases were ordered — all malignant first, as happens when a registry is exported sorted by diagnosis — unshuffled folds would be catastrophic, which is why a random shuffle with a fixed seed is used whenever row order could carry information. The chunk builds the folds with index arithmetic, checks them against the table, and applies the procedure to the 683 breast-cancer cases.



In [ ]:
# 5-fold cross-validation by hand: index arithmetic only, no library splitter.
kfold_indices <- function(n, k, shuffle = FALSE, seed = 0) {
  if (shuffle) { set.seed(seed); ord <- sample(n) } else ord <- seq_len(n)
  blocks <- split(ord, cut(seq_len(n), breaks = k, labels = FALSE))   # k nearly equal blocks
  lapply(blocks, function(f) list(train = setdiff(ord, f), test = sort(f)))
}
# (a) the toy example of the table: n = 10, k = 5, no shuffling
toy <- kfold_indices(10, 5)
for (j in seq_along(toy)) cat(sprintf("fold %d: test %-7s train %s\n", j,
    paste(toy[[j]]$test, collapse = " "), paste(toy[[j]]$train, collapse = " ")))

# (b) the same procedure on the breast-cancer data, with shuffling, for the 5-NN model
folds5 <- kfold_indices(nrow(X), 5, shuffle = TRUE, seed = 0)
fold_acc <- sapply(folds5, function(f) mean(knn(Xs[f$train, ], Xs[f$test, ], factor(y_pos[f$train]), k = 5) == y_pos[f$test]))
cat("\nbreast cancer, 5 folds of", sapply(folds5, function(f) length(f$test)), "cases\n")
cat("fold accuracies:", round(fold_acc, 3), " ->  CV accuracy =", round(mean(fold_acc), 3),
    sprintf("(SD across folds %.3f)", sd(fold_acc)), "\n")



Part (a) reproduces the table exactly. Part (b) gives a five-fold CV accuracy for 5-NN of about 0.96–0.97, with fold-to-fold differences of one or two points — the spread of Figure 2, now visible within one run, and itself a finding: a model whose fold scores range from 0.80 to 0.99 would be unstable.

### 3.3 The resampling schemes

A convenience function such as scikit-learn's <code>cross_val_score</code><sup class="c"><a href="#r22">[22]</a></sup> wraps this loop; in R the same effect is a one-line <code>sapply</code> over an index list, and the <code>caret</code> package<sup class="c"><a href="#r34">[34]</a></sup> provides a packaged equivalent (<code>trainControl</code>, <code>train</code>) when one prefers it. The schemes differ only in how the index list is built:

- **Plain k-fold** — the version of Section 3.2; suitable for large, balanced datasets, where every fold is representative by sheer size. Cost: \(k\) fits.
- **Stratified k-fold** — each fold preserves the class proportions (Section 2.2); the choice for small or imbalanced datasets, and the default for classification in this course. Cost: \(k\) fits.
- **Repeated stratified k-fold** — the k-fold procedure repeated \(r\) times with different shuffles, \(kr\) scores averaged. Reduces the variance due to the particular partition (not the variance due to having only \(n\) cases, which nothing reduces). Cost: \(kr\) fits.
- **Leave-one-out** — \(k = n\): each case is predicted by a model trained on all the others. Nearly unbiased for the error at size \(n-1\), but it costs \(n\) fits and, because the \(n\) training sets are almost identical, its estimate has high variance;<sup class="c"><a href="#r18">[18]</a></sup> suitable only for very small datasets.
- **Grouped k-fold** — cases from the same group (patient, hospital, image series) never appear in both training and test folds (Section 7.3).
- **Repeated hold-out (shuffle-split)** — as in Figure 2; test sets overlap between repeats.

Raschka's tutorial surveys these schemes, and the bootstrap alternatives omitted here, with worked code.<sup class="c"><a href="#r19">[19]</a></sup> The chunk compares them on the same model and data, with the number of fits each one costs.



In [ ]:
# The same model (5-NN) evaluated with several resampling schemes; mean, SD across folds and number of fits.
strat_folds <- function(y, k, seed = 0) {                    # assign each case a fold, preserving class proportions
  set.seed(seed); fold <- integer(length(y))
  for (cl in unique(y)) { i <- sample(which(y == cl)); fold[i] <- (seq_along(i) - 1) %% k + 1 }
  fold
}
strat_kfold <- function(y, k, seed = 0) { f <- strat_folds(y, k, seed)
  lapply(1:k, function(j) list(train = which(f != j), test = which(f == j))) }
acc_of <- function(splits) sapply(splits, function(f) mean(knn(Xs[f$train, ], Xs[f$test, ], factor(y_pos[f$train]), k = 5) == y_pos[f$test]))

schemes <- list(
  "KFold(5, shuffle)"             = kfold_indices(nrow(X), 5, shuffle = TRUE, seed = 0),
  "StratifiedKFold(5)"            = strat_kfold(y_pos, 5, 0),
  "StratifiedKFold(10)"           = strat_kfold(y_pos, 10, 0),
  "RepeatedStratifiedKFold(5x10)" = do.call(c, lapply(1:10, function(r) strat_kfold(y_pos, 5, r))),
  "ShuffleSplit(20 x 30%)"        = lapply(1:20, function(s) { set.seed(s); id <- sample(nrow(X), round(0.7 * nrow(X))); list(train = id, test = setdiff(seq_len(nrow(X)), id)) }),
  "LeaveOneOut (683 fits)"        = lapply(seq_len(nrow(X)), function(i) list(train = setdiff(seq_len(nrow(X)), i), test = i)))
rows <- lapply(names(schemes), function(nm) { t0 <- Sys.time(); s <- acc_of(schemes[[nm]])
  data.frame(scheme = nm, `mean accuracy` = mean(s), `SD across folds` = sd(s), fits = length(s),
             seconds = round(as.numeric(Sys.time() - t0), 2), check.names = FALSE) })
r <- do.call(rbind, rows); rownames(r) <- NULL; r[, 2:3] <- round(r[, 2:3], 3); r



All the schemes agree on the mean to within a few thousandths: with 683 cases and a stable model the estimate is not sensitive to the scheme. The SD of leave-one-out is meaningless as uncertainty (each fold is one case scored 0 or 1); the cost column is the price list — instantaneous for k-NN, decisive for the networks of Units 6 and 8, where one fit can take hours. The figure makes the schemes visible: each row a scheme, each column a case (sorted by class), colour the fold in which the case is tested.



In [ ]:
# Which fold tests which case? A visual comparison of four schemes on a small sorted sample.
n_small <- 40; y_small <- c(rep(1, 14), rep(0, 26)); g_small <- rep(1:10, each = 4)   # 14 malignant then 26 benign; 10 patients x 4 records
assign_fold <- function(scheme) {
  if (scheme == "kf")  return(cut(1:n_small, 4, labels = FALSE))
  if (scheme == "kfs") { set.seed(0); ord <- sample(n_small); fo <- integer(n_small); fo[ord] <- cut(1:n_small, 4, labels = FALSE); return(fo) }
  if (scheme == "skf") return(strat_folds(y_small, 4, 0))
  if (scheme == "gkf") { set.seed(0); pf <- sample(rep(1:4, length.out = 10)); return(pf[g_small]) }
}
nms <- c("KFold(4), no shuffle" = "kf", "KFold(4), shuffle" = "kfs", "StratifiedKFold(4)" = "skf", "GroupKFold(4), 10 patients" = "gkf")
cols4 <- c(BLUE, ORANGE, GREEN, GREY)
par(mfrow = c(5, 1), mar = c(0.4, 11, 0.4, 1), oma = c(3, 0, 2, 0))
for (nm in names(nms)) { image(1:n_small, 1, matrix(assign_fold(nms[nm]), ncol = 1), col = cols4, axes = FALSE, xlab = "", ylab = "")
  mtext(nm, side = 2, las = 1, cex = 0.72, line = 0.5) }
image(1:n_small, 1, matrix(y_small, ncol = 1), col = c("grey80", "grey30"), axes = FALSE, xlab = "", ylab = "")
mtext("class (dark = malignant)", side = 2, las = 1, cex = 0.68, line = 0.5)
mtext("case index (sorted by class);  colour = fold in which the case is tested", side = 1, outer = TRUE, line = 1, cex = 0.78)
mtext("Fold assignment of 40 cases under four schemes", side = 3, outer = TRUE, font = 2, cex = 0.9)



<p class="caption"><b>Figure 3.</b> Fold membership of 40 cases (14 malignant then 26 benign) under four schemes. Unshuffled k-fold puts the first cases — all malignant — in one test fold. Shuffled k-fold mixes the classes but counts per fold still vary. Stratified k-fold gives every fold the same class composition. Grouped k-fold keeps the four records of each patient together, so no patient spans the train/test boundary (Section 7.3).</p>

### 3.4 Nested cross-validation and the optimism of selecting on the test set

Cross-validation is also the standard tool for <strong>choosing hyperparameters</strong>: compute the CV score of each candidate setting and keep the best. The danger is to report that best score as the model's performance: it is the maximum of many noisy estimates and biased upwards, the more so the more candidates. Varma and Simon showed on small genomic datasets that the bias can exceed 20 percentage points; the remedy is <strong>nested cross-validation</strong>: an outer loop holds out a test fold, and inside each outer training set a complete inner CV selects the hyperparameters, so the outer test folds are never used for selection.<sup class="c"><a href="#r3">[3]</a></sup> Cawley and Talbot give the general theory.<sup class="c"><a href="#r23">[23]</a></sup>

The cleanest demonstration uses data with <em>no signal at all</em> — 100 cases, 20 noise features, random labels — where any honest estimate must be about 0.50. The chunk tunes a support-vector machine (Unit 3) over a 5 × 5 grid and reports (i) the best inner CV score, which a careless analysis would publish, and (ii) the nested estimate.



In [ ]:
# Optimistic bias of hyperparameter selection, measured on pure noise (true accuracy = 0.5 by construction).
Cs <- 10^seq(-2, 2, length.out = 5); gs <- 10^seq(-3, 1, length.out = 5)    # 25 candidate settings
grid_best <- function(Xd, yd, seed) {                                      # best 5-fold CV accuracy over the grid
  fi <- strat_folds(as.integer(yd), 5, seed); best <- -1
  for (C in Cs) for (g in gs) {
    a <- sapply(1:5, function(j) { tr <- fi != j
      mean(predict(svm(Xd[tr, ], yd[tr], cost = C, gamma = g, kernel = "radial"), Xd[!tr, ]) == yd[!tr]) })
    if (mean(a) > best) { best <- mean(a); bp <- c(C, g) } }
  list(best = best, par = bp)
}
naive <- nested <- numeric(6)
for (rep in 1:6) {                                                         # 6 independent noise datasets
  set.seed(rep); Xn <- matrix(rnorm(100 * 20), 100, 20); yn <- factor(sample(0:1, 100, replace = TRUE))
  naive[rep] <- grid_best(Xn, yn, rep)$best                               # best of 25 inner-CV scores: biased
  of <- strat_folds(as.integer(yn), 5, 100 + rep)                         # outer folds
  oacc <- sapply(1:5, function(j) { tr <- of != j; bp <- grid_best(Xn[tr, ], yn[tr], 1000 + j)$par
    mean(predict(svm(Xn[tr, ], yn[tr], cost = bp[1], gamma = bp[2], kernel = "radial"), Xn[!tr, ]) == yn[!tr]) })
  nested[rep] <- mean(oacc)                                               # honest
}
cat(sprintf("best inner-CV score (naive): mean %.3f  [%.3f, %.3f]\n", mean(naive), min(naive), max(naive)))
cat(sprintf("nested-CV estimate:          mean %.3f  [%.3f, %.3f]\n", mean(nested), min(nested), max(nested)))
cat(sprintf("mean optimism of the naive number: %+.3f  (true accuracy is 0.500)\n", mean(naive) - mean(nested)))



On data with nothing to learn, the best-of-25 inner score averages several points above 0.5 — a signal manufactured entirely by selection — while the nested estimate stays near chance. With 100 cases the effect is modest; with the 20–50 cases per class of a typical biomarker study and a grid of hundreds of settings, Varma and Simon's 20-point figure is reached easily, and a useless model is published as a useful one.<sup class="c"><a href="#r3">[3]</a></sup>

<div class="box key"><h4>Key idea: one level of looking, one level of holding out</h4><p>Every decision taken by looking at a score — which model, hyperparameters, features, threshold — must be taken on data that are not then used to report the score. Nested CV automates this for hyperparameters; the same logic applies to feature selection (Section 7.1) and to manual exploration: if you tried twenty things and report the best, the number is optimistic, and a fresh test set is the only remedy.</p></div>

<div class="box"><h4>Exercise 2</h4><p>Enlarge the grid to 10 × 10 settings and reduce the sample to 50 cases: does the optimism grow or shrink? Then replace the noise by the breast-cancer data: with real signal, how large is the gap between the naive and the nested number?</p></div>

## 4. Overfitting and the bias–variance trade-off

### 4.1 Complexity curves

The <strong>complexity</strong> of a model is, loosely, the number of free parameters it can adjust: the degree of a polynomial (degree \(d\) has \(d+1\) coefficients), the depth of a tree, the number of weights in a network. A model <strong>underfits</strong> when it is too simple to represent the regularity in the data — a straight line through a curved relationship — and performs poorly on training and test data alike. It <strong>overfits</strong> when it is complex enough to reproduce the accidents of the training sample — label noise, chance correlations — rather than the regularity; the signature is a growing gap between training and held-out performance as flexibility increases. For k-nearest neighbours (Unit 3) flexibility is set by \(k\): \(k = 1\) follows every training case individually, \(k = n\) predicts the majority class for everyone. The chunk sweeps \(k\) and records training and ten-fold CV accuracy.



In [ ]:
# Complexity curve: training accuracy and cross-validated accuracy of k-NN as a function of k.
ks <- c(1, 3, 5, 7, 9, 11, 15, 21, 31, 41, 61, 81, 101, 151, 201, 301)
cv10 <- strat_kfold(y_pos, 10, 0)
train_acc <- sapply(ks, function(k) mean(knn(Xs, Xs, factor(y_pos), k = k) == y_pos))
cv_acc <- sapply(ks, function(k) mean(sapply(cv10, function(f) mean(knn(Xs[f$train, ], Xs[f$test, ], factor(y_pos[f$train]), k = k) == y_pos[f$test]))))
best <- ks[which.max(cv_acc)]
cat(sprintf("best k by CV: %d (CV accuracy %.3f); at k=1: train %.3f / CV %.3f; at k=301: train %.3f / CV %.3f\n",
            best, max(cv_acc), train_acc[1], cv_acc[1], tail(train_acc, 1), tail(cv_acc, 1)))
plot(ks, train_acc, type = "o", pch = 19, col = GREY, log = "x", xlim = rev(range(ks)), ylim = c(0.80, 1.01),
     xlab = "k (number of neighbours) - flexibility increases to the right", ylab = "accuracy",
     main = "k-NN on the breast-cancer data: training vs cross-validated accuracy")
lines(ks, cv_acc, type = "o", pch = 15, col = BLUE); abline(v = best, col = ORANGE, lty = 2)
legend("bottom", bty = "n", cex = 0.8, pch = c(19, 15, NA), lty = c(1, 1, 2), col = c(GREY, BLUE, ORANGE),
       legend = c("training accuracy (resubstitution)", "10-fold CV accuracy", sprintf("best k by CV = %d", best)))



<p class="caption"><b>Figure 4.</b> Training and cross-validated accuracy of k-NN against \(k\) (flexibility increasing to the right). At the right edge training accuracy is 1.000 and CV accuracy lower: the model memorises. At the left edge both fall together towards the majority-class rate of 0.650 (444 benign of 683): underfitting. The CV curve peaks in between, where the curves are close. The training curve is monotone and uninformative; only the CV curve can locate the optimum.</p>

### 4.2 Learning curves

A second diagnostic plots performance against the <em>size of the training sample</em>. The training score falls as \(n\) grows (a larger sample is harder to memorise) and the validation score rises (there is more to learn from). The gap at the largest \(n\) is the remaining variance; the level at which the curves converge is the remaining bias. A learning curve answers the most expensive question in applied work — would collecting more patients help? — before the patients are collected.



In [ ]:
# Learning curves for two models of different flexibility: training score and CV score against training-set size.
dff <- data.frame(Xs, y = y_pos)
pred_logit <- function(sub, test) { fit <- suppressWarnings(glm(y ~ ., dff[sub, ], family = binomial)); as.integer(predict(fit, dff[test, ], type = "response") >= 0.5) }
pred_1nn   <- function(sub, test) as.integer(as.character(knn(Xs[sub, , drop = FALSE], Xs[test, , drop = FALSE], factor(y_pos[sub]), k = 1)))
fracs <- seq(0.1, 1, length.out = 8); cv <- strat_kfold(y_pos, 5, 0)
learn <- function(predf) t(sapply(fracs, function(fr) {
  tr <- cv_ <- numeric(length(cv))
  for (i in seq_along(cv)) { f <- cv[[i]]; m <- max(15, round(fr * length(f$train))); set.seed(i); sub <- f$train[sample(length(f$train), m)]
    cv_[i] <- mean(predf(sub, f$test) == y_pos[f$test]); tr[i] <- mean(predf(sub, sub) == y_pos[sub]) }
  c(size = round(fr * length(cv[[1]]$train)), train = mean(tr), cv = mean(cv_)) }))
par(mfrow = c(1, 2))
for (nm in c("logistic regression (Unit 2)", "1-nearest neighbour")) {
  L <- if (grepl("logistic", nm)) learn(pred_logit) else learn(pred_1nn)
  plot(L[, "size"], L[, "train"], type = "o", pch = 19, col = GREY, ylim = c(0.85, 1.005), xlab = "number of training cases", ylab = "accuracy", main = nm)
  lines(L[, "size"], L[, "cv"], type = "o", pch = 15, col = BLUE)
  if (grepl("logistic", nm)) legend("bottomright", bty = "n", cex = 0.8, pch = c(19, 15), col = c(GREY, BLUE), legend = c("training score", "CV score"))
  cat(sprintf("%-30s at n=%d: train %.3f, CV %.3f, gap %.3f\n", nm, L[nrow(L), "size"], L[nrow(L), "train"], L[nrow(L), "cv"], L[nrow(L), "train"] - L[nrow(L), "cv"]))
}



<p class="caption"><b>Figure 5.</b> Learning curves. Left: logistic regression, whose training and CV curves converge to within about one point at full size — a low-variance model that has extracted most of what these data contain, so more patients would help little. Right: 1-NN, whose training score stays at 1.000 while the CV score plateaus below it — a persistent gap that more data narrows only slowly.</p>

### 4.3 The bias–variance decomposition

The two diagnostics have one theory behind them. For a regression problem \(y = f(x) + \varepsilon\) with noise variance \(\sigma^2\) and a procedure that returns \(\hat{f}_D(x)\) from a training sample \(D\), the expected squared error at \(x\), over samples and over the noise of a new observation, decomposes exactly as

<div class="eq">\[ \mathbb{E}\big[(y - \hat{f}_D(x))^2\big] = \underbrace{\big(f(x) - \mathbb{E}_D[\hat{f}_D(x)]\big)^2}_{\text{bias}^2} + \underbrace{\mathbb{E}_D\big[(\hat{f}_D(x) - \mathbb{E}_D[\hat{f}_D(x)])^2\big]}_{\text{variance}} + \underbrace{\sigma^2}_{\text{irreducible noise}}, \]
<span class="where">where \(\mathbb{E}_D\) is over training samples. <strong>Bias</strong> is error from erroneous assumptions in the model — a straight line assumed where the truth is curved; it is large for rigid models, and high bias produces <em>underfitting</em>. <strong>Variance</strong> is the sensitivity of the fit to small fluctuations of the training set; it is large for flexible models, and high variance produces <em>overfitting</em>. The noise term is the floor no model goes below. The decomposition comes from Geman, Bienenstock and Doursat's 1992 analysis of neural networks;<sup class="c"><a href="#r15">[15]</a></sup> chapter 7 of Hastie, Tibshirani and Friedman develops it for the methods of this course.<sup class="c"><a href="#r18">[18]</a></sup></span></div>

More flexibility lowers bias and raises variance; the test error, their sum plus noise, is U-shaped, and its minimum is the right complexity. Figure 4 was this U-shape seen through CV accuracy. The decomposition cannot be computed on real data — \(f\) is unknown and only one sample exists — but it can be simulated: 300 training samples of 20 points from \(y = \sin(2\pi x) + \varepsilon\), \(\sigma = 0.3\), a polynomial of each degree fitted to every sample, bias\(^2\) and variance measured across the 300 fits.



In [ ]:
# Bias-variance simulation: 300 training samples, polynomial fits of degree 0-12 to y = sin(2 pi x) + noise.
set.seed(0)
f_true <- function(x) sin(2 * pi * x); n_pts <- 20; sigma <- 0.3; R <- 300
x_fixed <- seq(0, 1, length.out = n_pts)                              # fixed design: same x in every sample
Y <- matrix(f_true(x_fixed), R, n_pts, byrow = TRUE) + matrix(rnorm(R * n_pts, 0, sigma), R, n_pts)
degrees <- 0:12
fitmat <- function(dg) t(sapply(1:R, function(r) if (dg == 0) rep(mean(Y[r, ]), n_pts) else predict(lm(Y[r, ] ~ poly(x_fixed, dg, raw = TRUE)))))
fits <- lapply(degrees, fitmat); names(fits) <- degrees
Yfresh <- matrix(f_true(x_fixed), R, n_pts, byrow = TRUE) + matrix(rnorm(R * n_pts, 0, sigma), R, n_pts)   # fresh noisy sample
decomp <- data.frame(
  degree = degrees,
  `bias^2` = sapply(fits, function(fm) mean((colMeans(fm) - f_true(x_fixed))^2)),
  variance = sapply(fits, function(fm) mean(apply(fm, 2, var))),
  noise = sigma^2, check.names = FALSE)
decomp$`bias^2 + variance + noise` <- decomp$`bias^2` + decomp$variance + decomp$noise
decomp$`empirical test MSE` <- sapply(fits, function(fm) mean((fm - Yfresh)^2))
round(decomp[c(1, 2, 3, 4, 5, 7, 10, 13), ], 4)



The table is the decomposition, degree by degree. At degree 0 (a horizontal line) bias\(^2\) is large and variance tiny; by degree 3 bias\(^2\) has almost vanished, because a cubic follows one period of a sine closely; from there on variance grows with every coefficient while bias\(^2\) stays near zero. The column "bias\(^2\) + variance + noise" matches the mean squared error against fresh noisy observations to within simulation error — the identity holds. The figure shows the same as curves.



In [ ]:
# Left: 30 of the 300 fits for three degrees; their mean (thick) and the truth (dashed). Right: the decomposition.
par(mfrow = c(1, 2))
plot(NA, xlim = c(0, 1), ylim = c(-2, 2), xlab = "x", ylab = "y", main = "Rigid fits agree and miss; flexible scatter")
for (dc in list(c(1, GREY), c(3, BLUE), c(12, ORANGE))) { dg <- as.integer(dc[1]); col <- dc[2]
  for (r in 1:30) lines(x_fixed, fits[[as.character(dg)]][r, ], col = adjustcolor(col, 0.30), lwd = 0.6)
  lines(x_fixed, colMeans(fits[[as.character(dg)]]), col = col, lwd = 2.4) }
lines(x_fixed, f_true(x_fixed), lty = 2, lwd = 1.4); points(x_fixed, Y[1, ], pch = 19, cex = 0.6)
legend("topright", bty = "n", cex = 0.72, lwd = 2, col = c(GREY, BLUE, ORANGE, "black"),
       lty = c(1, 1, 1, 2), legend = c("degree 1 (mean)", "degree 3 (mean)", "degree 12 (mean)", "truth sin(2 pi x)"))
plot(decomp$degree, decomp$`bias^2`, type = "o", pch = 19, col = GREY, ylim = c(0, 0.6),
     xlab = "polynomial degree (flexibility)", ylab = "mean squared error", main = "The U-shaped test error")
lines(decomp$degree, decomp$variance, type = "o", pch = 15, col = ORANGE)
lines(decomp$degree, decomp$`bias^2 + variance + noise`, type = "o", pch = 17, col = BLUE)
abline(h = sigma^2, lty = 3)
legend("top", bty = "n", cex = 0.78, pch = c(19, 15, 17, NA), lty = c(1, 1, 1, 3), col = c(GREY, ORANGE, BLUE, "black"),
       legend = c("bias^2", "variance", "total = test MSE", sprintf("noise floor = %.2f", sigma^2)))



<p class="caption"><b>Figure 6.</b> Left: thirty fits of each degree to thirty training samples. The straight lines (degree 1) agree with one another and all miss the sine — pure bias. The degree-12 curves follow the truth on average but each one wanders — pure variance. The cubics do both jobs adequately. Right: bias\(^2\) falls and variance rises with degree; their sum is minimal at a low degree and never reaches the noise floor \(\sigma^2 = 0.09\).</p>

<div class="box bridge"><h4>Bridge to the units</h4><p>Every method in the course has a knob on this curve: \(k\) in k-NN and the kernel width of an SVM (Unit 3), tree depth and number of trees (Unit 4), layers and regularisation of a network (Units 6 and 8), degree or penalty of a regression (Unit 2). The knob is <em>always</em> set with validation data or cross-validation, never with the training score, which only ever rewards flexibility.</p></div>

<div class="box"><h4>Exercise 3</h4><p>Change <code>n_pts</code> from 20 to 100 in the simulation and re-run both chunks. The variance curve flattens and the optimal degree moves to the right: with more data, a more flexible model becomes affordable. Then set <code>sigma = 0.6</code>. What happens to the noise floor, and to the degree at which the total error is minimal?</p></div>

### 4.4 The curse of dimensionality and the parameters-to-observations ratio

Complexity also grows with the number of features. Each feature adds a parameter to a linear model and a dimension to the space in which k-NN measures distances, and in high dimensions a fixed sample becomes sparse: every case is far from every other and "nearest" neighbours are no longer near — the <strong>curse of dimensionality</strong>.<sup class="c"><a href="#r18">[18]</a></sup> The practical measure is the parameters-to-observations ratio \(p/n\). With \(p \geq n\) a linear model can fit any labels perfectly, so its training score becomes meaningless; a rule of thumb for clinical prediction models asks for at least ten to twenty events per candidate parameter.<sup class="c"><a href="#r27">[27]</a></sup> The chunk appends pure-noise features to the nine real ones.



In [ ]:
# Curse of dimensionality: noise features appended to the nine real ones (n = 683 fixed); training vs CV accuracy.
set.seed(0); cv5 <- strat_kfold(y_pos, 5, 0); rows <- list()
for (p_noise in c(0, 30, 100, 300, 1000)) {
  Xp <- scale(cbind(X, matrix(rnorm(nrow(X) * p_noise), nrow(X), p_noise)))
  dfp <- data.frame(Xp, y = y_pos)
  knn_tr <- mean(knn(Xp, Xp, factor(y_pos), k = 5) == y_pos)
  knn_cv <- mean(sapply(cv5, function(f) mean(knn(Xp[f$train, ], Xp[f$test, ], factor(y_pos[f$train]), k = 5) == y_pos[f$test])))
  fit <- suppressWarnings(glm(y ~ ., dfp, family = binomial)); lr_tr <- mean((predict(fit, type = "response") >= 0.5) == y_pos)
  lr_cv <- mean(sapply(cv5, function(f) { fit <- suppressWarnings(glm(y ~ ., dfp[f$train, ], family = binomial))
    mean((predict(fit, dfp[f$test, ], type = "response") >= 0.5) == y_pos[f$test]) }))
  rows[[length(rows) + 1]] <- data.frame(`noise added` = p_noise, `p/n` = round(ncol(Xp) / nrow(X), 2),
    `5NN train` = knn_tr, `5NN CV` = knn_cv, `logit train` = lr_tr, `logit CV` = lr_cv, check.names = FALSE)
}
curse <- do.call(rbind, rows); rownames(curse) <- NULL; curse[, 3:6] <- round(curse[, 3:6], 3); curse



The information is identical in every row — the same nine real features — and only useless columns were added. As \(p/n\) passes 1, the training accuracy of logistic regression reaches 1.000 (it can now separate any labelling) while its CV accuracy declines; for k-NN both scores fall, because distances become dominated by noise. Feature selection and regularisation (Units 2–4) exist to fight this, and must themselves be cross-validated (Section 7.1).

## 5. Metrics for classification

### 5.1 The confusion matrix

Every metric for a binary classifier is a function of four counts. Fix which class is <strong>positive</strong> — in medicine, the condition being detected — and cross-tabulate true against predicted class:

| | predicted positive | predicted negative | |
|---|---|---|---|
| **truly positive** | TP (true positive) | FN (false negative, <em>missed case</em>) | \(P = \mathrm{TP} + \mathrm{FN}\) |
| **truly negative** | FP (false positive, <em>false alarm</em>) | TN (true negative) | \(N = \mathrm{FP} + \mathrm{TN}\) |

From these, the metrics the course uses, with machine-learning and clinical names side by side (Altman and Bland's BMJ notes remain the clearest short account of the clinical vocabulary<sup class="c"><a href="#r16">[16]</a>,<a href="#r17">[17]</a></sup>):

| Metric | Formula | Clinical name | Reads as |
|---|---|---|---|
| Accuracy | \((\mathrm{TP}+\mathrm{TN})/(P+N)\) | — | fraction of all cases classified correctly |
| Sensitivity = **recall** = TPR | \(\mathrm{TP}/P\) | sensitivity | of the diseased, the fraction detected |
| Specificity = TNR | \(\mathrm{TN}/N\) | specificity | of the healthy, the fraction cleared |
| False-positive rate (FPR) | \(\mathrm{FP}/N = 1 - \text{specificity}\) | — | of the healthy, the fraction falsely alarmed |
| **Precision** = PPV | \(\mathrm{TP}/(\mathrm{TP}+\mathrm{FP})\) | PPV | of the predicted positives, the fraction truly ill |
| Negative predictive value (NPV) | \(\mathrm{TN}/(\mathrm{TN}+\mathrm{FN})\) | NPV | of the predicted negatives, the fraction truly healthy |
| F1 score | \(2\,\mathrm{TP}/(2\,\mathrm{TP}+\mathrm{FP}+\mathrm{FN})\) | — | harmonic mean of precision and recall; ignores TN |
| Balanced accuracy | \((\text{sensitivity} + \text{specificity})/2\) | — | accuracy if both classes were equally frequent<sup class="c"><a href="#r29">[29]</a></sup> |
| Matthews correlation coefficient | \(\dfrac{\mathrm{TP}\cdot\mathrm{TN} - \mathrm{FP}\cdot\mathrm{FN}}{\sqrt{(\mathrm{TP}+\mathrm{FP})(\mathrm{TP}+\mathrm{FN})(\mathrm{TN}+\mathrm{FP})(\mathrm{TN}+\mathrm{FN})}}\) | — | correlation of true and predicted class, −1 to +1, 0 at chance<sup class="c"><a href="#r7">[7]</a></sup> |

Sensitivity and specificity are properties of the <em>test</em>, computed within the diseased and within the healthy, so they do not change when the disease becomes rarer. PPV and NPV are properties of the test <em>in a population</em> and depend on how common the disease is (Section 5.3).

### 5.2 Worked example by hand

A screening test is applied to 100 patients, of whom 20 have the disease. It flags 23 patients: 18 who have the disease and 5 who do not. So \(\mathrm{TP} = 18\), \(\mathrm{FN} = 2\), \(\mathrm{FP} = 5\), \(\mathrm{TN} = 75\).

- Accuracy \(= (18 + 75)/100 = 0.930\)
- Sensitivity \(= 18/20 = 0.900\); specificity \(= 75/80 = 0.9375\)
- PPV \(= 18/23 \approx 0.783\); NPV \(= 75/77 \approx 0.974\)
- F1 \(= 2 \cdot 18/(2 \cdot 18 + 5 + 2) = 36/43 \approx 0.837\)
- Balanced accuracy \(= (0.900 + 0.9375)/2 \approx 0.919\)
- MCC \(= (18 \cdot 75 - 5 \cdot 2)/\sqrt{23 \cdot 20 \cdot 80 \cdot 77} = 1340/\sqrt{2\,833\,600} \approx 0.796\)

The chunk computes the same numbers from the four counts and, for a cross-check, rebuilds the 100 patients as label/prediction vectors and recovers the counts with <code>table()</code>.



In [ ]:
# The hand-worked confusion matrix, as code: metrics from the four counts, and a table() cross-check.
metrics_from_counts <- function(tp, fn, fp, tn) {
  tp <- as.numeric(tp); fn <- as.numeric(fn); fp <- as.numeric(fp); tn <- as.numeric(tn)   # avoid integer overflow in MCC
  P <- tp + fn; N <- fp + tn; sens <- tp / P; spec <- tn / N
  dv <- function(a, b) if (b > 0) a / b else 0                       # a ratio with an empty denominator is reported as 0
  mcc_den <- sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
  c(accuracy = (tp + tn) / (P + N), `sensitivity (recall)` = sens, specificity = spec,
    `precision (PPV)` = dv(tp, tp + fp), NPV = dv(tn, tn + fn), F1 = dv(2 * tp, 2 * tp + fp + fn),
    `balanced accuracy` = (sens + spec) / 2, MCC = dv(tp * tn - fp * fn, mcc_den))
}
# rebuild the 100 patients of the example as label/prediction vectors (1 = disease)
yt <- c(rep(1, 20), rep(0, 80)); yp <- c(rep(1, 18), rep(0, 2), rep(1, 5), rep(0, 75))
cm <- table(truth = factor(yt, c(1, 0)), predicted = factor(yp, c(1, 0)))
print(cm)
tp <- cm["1", "1"]; fn <- cm["1", "0"]; fp <- cm["0", "1"]; tn <- cm["0", "0"]
cat(sprintf("\nTP=%d FN=%d FP=%d TN=%d\n", tp, fn, fp, tn))
round(metrics_from_counts(tp, fn, fp, tn), 4)



The computed vector agrees with the hand calculation to the last digit. Two practical remarks. A confusion table is only unambiguous once the factor levels are fixed (here <code>factor(.., c(1,0))</code> puts the positive class first, giving the epidemiological [[TP, FN], [FP, TN]] layout); the default alphabetical/numeric order would place TN in the top-left corner, the classic source of swapped sensitivity and specificity. And every class-dependent metric has a mirror image: specificity is the sensitivity of the negative class, NPV its precision.

<div class="box warn"><h4>Which class is positive?</h4><p>The <code>mlbench</code> Wisconsin data code the label as a factor with levels <code>benign</code> then <code>malignant</code>, so <code>as.integer(Class)</code> would make benign = 1 — the opposite of the clinical convention. Every chunk here defines <code>y_pos = as.integer(Class == "malignant")</code> so that 1 is the condition to detect. <em>Which label is 1?</em> should be the first line of any evaluation you write.</p></div>



In [ ]:
# Metrics of a logistic-regression classifier on the breast-cancer data, positive = malignant, via 10-fold CV predictions.
cvprob_glm <- function(Xm, y, k = 10, seed = 0) {           # out-of-fold P(malignant) for each case
  f <- strat_folds(y, k, seed); p <- numeric(length(y)); df <- data.frame(Xm, y = y)
  for (j in 1:k) { tr <- f != j; fit <- suppressWarnings(glm(y ~ ., df[tr, ], family = binomial))
    p[!tr] <- predict(fit, df[!tr, ], type = "response") }
  p
}
prob_cv <- cvprob_glm(as.matrix(Xs), y_pos)                 # reused throughout Sections 5-8
pred_cv <- as.integer(prob_cv >= 0.5)
tp <- sum(pred_cv == 1 & y_pos == 1); fn <- sum(pred_cv == 0 & y_pos == 1)
fp <- sum(pred_cv == 1 & y_pos == 0); tn <- sum(pred_cv == 0 & y_pos == 0)
cm <- matrix(c(tp, fp, fn, tn), 2, 2, dimnames = list(c("truly malignant", "truly benign"), c("pred. malignant", "pred. benign")))
print(cm)
round(metrics_from_counts(tp, fn, fp, tn), 3)



In clinical terms: of the 239 malignant tumours a handful are missed (FN) — each a delayed diagnosis — and of the 444 benign ones a handful are flagged (FP), each an unnecessary biopsy. Sensitivity and specificity are both in the range 0.95–0.98, and because the classes are of comparable size, accuracy, balanced accuracy, F1 and MCC agree. That agreement is a property of this dataset's balance, not of the metrics.

### 5.3 Imbalance: when accuracy fails, and why PPV depends on prevalence

Most screening problems are <strong>imbalanced</strong>: the class of interest, the <strong>minority class</strong>, is rare and the <strong>majority class</strong> dominates. The degree of imbalance is conventionally described by the share of the minority class:

| Degree of imbalance | Minority class share | Example |
|---|---|---|
| Mild | 20–40 % | malignant tumours in the Wisconsin sample (35 %) |
| Moderate | 1–20 % | diabetic retinopathy in a screening programme |
| Extreme | < 1 % | a rare disease, fraud, a critical device fault |

The slide example of the lecture shows why accuracy fails here. A spam filter is applied to 10,000 e-mails of which 100 are spam. The filter that classifies <em>every</em> e-mail as "not spam" obtains \(\mathrm{TN} = 9{,}900\), \(\mathrm{FN} = 100\), \(\mathrm{TP} = 0\), \(\mathrm{FP} = 0\), hence an accuracy of \(9{,}900/10{,}000 = 99\,\%\) while detecting nothing. The simulation reproduces the situation with a medical reading: 20,000 cases, 1 % prevalence, a logistic regression, and the classifier that predicts "healthy" for everyone.



In [ ]:
# Imbalance: 1 % prevalence. Compare a real classifier with the trivial 'everyone is healthy' rule.
gen_imbalanced <- function(n, prev, p = 10, seed = 0, sep = 1.0) {     # class-conditional Gaussians, 5 informative features
  set.seed(seed); y <- as.integer(runif(n) < prev); Xm <- matrix(rnorm(n * p), n, p)
  Xm[y == 1, 1:5] <- Xm[y == 1, 1:5] + sep; list(X = Xm, y = y)
}
d <- gen_imbalanced(20000, 0.01, sep = 1.0)
fi <- strat_folds(d$y, 2, 0); tr <- fi == 1; dfi <- data.frame(d$X, y = d$y)
yi_te <- d$y[!tr]
fit_i <- suppressWarnings(glm(y ~ ., dfi[tr, ], family = binomial)); prob_imb <- predict(fit_i, dfi[!tr, ], type = "response")
cat(sprintf("prevalence of the positive class: %.3f  (%d positives among %d test cases)\n", mean(d$y), sum(yi_te), length(yi_te)))
res <- list()
for (nm in c("'everyone is healthy'", "logistic regression")) {
  pred <- if (grepl("healthy", nm)) rep(0L, length(yi_te)) else as.integer(prob_imb >= 0.5)
  tp <- sum(pred == 1 & yi_te == 1); fn <- sum(pred == 0 & yi_te == 1); fp <- sum(pred == 1 & yi_te == 0); tn <- sum(pred == 0 & yi_te == 0)
  m <- round(metrics_from_counts(tp, fn, fp, tn), 3); res[[nm]] <- c(m, `TP/FN/FP` = sprintf("%d/%d/%d", tp, fn, fp))
}
as.data.frame(do.call(rbind, res))



The trivial rule reaches an accuracy near 0.99 without detecting a single case — sensitivity, precision, F1 and MCC are zero and balanced accuracy is exactly 0.5, chance. The logistic regression has a very similar accuracy yet detects a share of the positives with high precision. Under imbalance, accuracy measures the prevalence, not the classifier; the informative metrics look inside each class (sensitivity, specificity, balanced accuracy, MCC) or inside the predicted-positive group (precision, F1). Chicco and Jurman argue for the MCC as the most informative single summary, because it is high only when all four cells are good;<sup class="c"><a href="#r8">[8]</a></sup> F1, which ignores true negatives, is the convention of information retrieval, where negatives are uninteresting and uncountable — rarely the case in medicine.

**Training on imbalanced data: downsampling and upweighting.** Try the original data first — logistic regression handles moderate imbalance correctly when the metrics are chosen well — and intervene only when the minority class is so rare that the model barely sees it. The standard intervention is to <em>downsample</em> the majority class and <em>upweight</em> the retained cases by the same factor, so that predicted probabilities still mean what they say. Downsampling without upweighting, or inverse-frequency ("balanced") weights, deliberately distorts those proportions: both raise sensitivity and both break calibration (Section 5.6). The chunk demonstrates the options on a 1 : 200 problem.



In [ ]:
# Downsampling the majority by 10 and upweighting it by 10 (weights) on a 1:200 problem; inverse-frequency weights for contrast.
d <- gen_imbalanced(60000, 0.005, sep = 1.0, seed = 0)
fi <- strat_folds(d$y, 2, 0); tr <- fi == 1; df <- data.frame(d$X, y = d$y)
dtr <- df[tr, ]; dte <- df[!tr, ]; yte <- d$y[!tr]
pos <- which(dtr$y == 1); neg <- which(dtr$y == 0); set.seed(0); keep <- c(pos, sample(neg, length(neg) %/% 10))
w_up <- ifelse(dtr$y[keep] == 1, 1, 10)                               # upweight retained majority cases by the same factor
w_bal <- ifelse(dtr$y == 1, 0.5 / mean(dtr$y), 0.5 / (1 - mean(dtr$y)))
report <- function(fit, ntr, ratio) { pr <- predict(fit, dte, type = "response"); pred <- as.integer(pr >= 0.5)
  tp <- sum(pred == 1 & yte == 1); fn <- sum(pred == 0 & yte == 1); fp <- sum(pred == 1 & yte == 0); tn <- sum(pred == 0 & yte == 0)
  round(c(`training cases` = ntr, ratio = ratio, sensitivity = tp / (tp + fn), specificity = tn / (tn + fp),
          PPV = tp / max(tp + fp, 1), AUC = as.numeric(auc(yte, pr, quiet = TRUE)), Brier = mean((pr - yte)^2),
          `mean pred. P` = mean(pr), `obs. prevalence` = mean(yte)), 4) }
tab <- rbind(
  "original data (1:200)"    = report(suppressWarnings(glm(y ~ ., dtr, family = binomial)), nrow(dtr), 200),
  "downsampled only (1:20)"  = report(suppressWarnings(glm(y ~ ., dtr[keep, ], family = binomial)), length(keep), 20),
  "downsampled + upweighted" = report(suppressWarnings(glm(y ~ ., dtr[keep, ], family = binomial, weights = w_up)), length(keep), 20),
  "inverse-frequency weights" = report(suppressWarnings(glm(y ~ ., dtr, family = binomial, weights = w_bal)), nrow(dtr), 200))
as.data.frame(tab)



Read the last two columns first. On the original data the mean predicted probability equals the observed prevalence (about 0.005). Downsampling alone multiplies it several-fold — every patient's risk is overstated — while the AUC barely moves; adding the weight of 10 brings it back to the prevalence with the same discrimination and a tenth of the majority cases. The inverse-frequency weighting lifts sensitivity dramatically, at the price of a far lower specificity and PPV and a much worse Brier score: it has turned a probability model into a ranking model, fine only if that was the intention.

**PPV depends on prevalence.** Sensitivity and specificity are fixed by the test, but the probability that a flagged patient is truly ill is not. Write \(\pi\) for the prevalence, \(\mathrm{Se}\) for sensitivity and \(\mathrm{Sp}\) for specificity. Bayes' theorem gives

<div class="eq">\[ \mathrm{PPV} = P(\text{disease} \mid \text{test}+) = \frac{\mathrm{Se}\,\pi}{\mathrm{Se}\,\pi + (1 - \mathrm{Sp})(1 - \pi)}, \qquad \mathrm{NPV} = \frac{\mathrm{Sp}\,(1-\pi)}{\mathrm{Sp}\,(1-\pi) + (1 - \mathrm{Se})\,\pi}. \]
<span class="where">The numerator is the rate of true positives in the population; the second term of the denominator is the rate of false alarms, proportional to the <em>healthy</em> fraction \(1 - \pi\). When the disease is rare the healthy are many, and even a small false-positive rate produces more false alarms than detections.<sup class="c"><a href="#r17">[17]</a></sup> Unit 3 derives Bayes' theorem; here it is applied.</span></div>



In [ ]:
# PPV as a function of prevalence for a fixed test (Bayes' theorem), for three levels of specificity.
ppv <- function(prev, se, sp) se * prev / (se * prev + (1 - sp) * (1 - prev))
prev <- 10^seq(-3, 0, length.out = 300)                              # prevalence from 0.1% to 100%
plot(prev * 100, ppv(prev, 0.90, 0.90), type = "l", log = "x", col = GREY, lwd = 2, ylim = c(0, 1),
     xlab = "prevalence of the condition (%)", ylab = "PPV: P(disease | test positive)", main = "The same test, in populations of different prevalence")
lines(prev * 100, ppv(prev, 0.90, 0.99), col = BLUE, lwd = 2); lines(prev * 100, ppv(prev, 0.90, 0.999), col = GREEN, lwd = 2)
for (p0 in c(1, 10, 50)) abline(v = p0, col = ORANGE, lty = 3)
legend("topleft", bty = "n", cex = 0.8, lwd = 2, col = c(GREY, BLUE, GREEN), legend = c("sens 0.90, spec 0.90", "sens 0.90, spec 0.99", "sens 0.90, spec 0.999"))
tab <- sapply(c(0.01, 0.10, 0.50), function(p0) sapply(c(0.90, 0.99, 0.999), function(sp) ppv(p0, 0.90, sp)))
dimnames(tab) <- list(c("specificity 0.90", "specificity 0.99", "specificity 0.999"), c("prevalence 1%", "prevalence 10%", "prevalence 50%"))
round(tab, 3)



<p class="caption"><b>Figure 7.</b> PPV against prevalence (logarithmic axis) for sensitivity 0.90 and three specificities; dotted lines at 1 %, 10 % and 50 %. A test with 90 % specificity — respectable where half the referred patients are ill — has a PPV of about 8 % when screening a population with 1 % prevalence: eleven of every twelve positives are false alarms.</p>

The lesson generalises: a model developed where 40 % of patients had the outcome will have a far lower PPV in primary care where 4 % have it, with no change in the model. Sensitivity and specificity transport between settings; predictive values do not, and a report that gives precision and recall without the prevalence of its test set has withheld the information needed to use it.

### 5.4 From scores to curves: ROC

Most classifiers output a <strong>score</strong> — a probability, a distance, a vote count — and the class is obtained by comparing it with a <strong>threshold</strong>; 0.5 is a convention, not a law. A higher threshold calls fewer cases positive, so precision tends to rise and recall falls; a lower one does the reverse. Precision and recall move in opposite directions under a change of threshold, which is why neither is reported without the other, and why F1 — their harmonic mean with equal weight — is a compromise rather than a resolution. The false-positive rate (FP/N) has the same dependence and is meaningful only when there are enough true negatives to estimate it, one reason the precision–recall curve is preferred when negatives are scarce.

The <strong>receiver operating characteristic</strong> (ROC) curve displays the whole family of classifiers obtained by sweeping the threshold: true-positive rate (sensitivity) against false-positive rate (1 − specificity). The area under it, the <strong>AUC</strong>, summarises ranking quality; Hanley and McNeil showed that it equals the probability that a randomly chosen positive receives a higher score than a randomly chosen negative.<sup class="c"><a href="#r4">[4]</a></sup> An AUC of 1.0 is a perfect ranking; 0.5 is random. Fawcett's tutorial is the standard reference for the construction and its pitfalls.<sup class="c"><a href="#r5">[5]</a></sup>

**By hand.** Six patients, three diseased (D) and three healthy (H), receive scores D 0.9, D 0.7, H 0.6, D 0.4, H 0.3, H 0.2. Sort by score and lower the threshold one case at a time; each step moves the operating point up (D) or right (H) by one third:

| threshold just below | case | TP | FP | TPR = TP/3 | FPR = FP/3 |
|---|---|---|---|---|---|
| (none positive) | — | 0 | 0 | 0 | 0 |
| 0.9 | D | 1 | 0 | 1/3 | 0 |
| 0.7 | D | 2 | 0 | 2/3 | 0 |
| 0.6 | H | 2 | 1 | 2/3 | 1/3 |
| 0.4 | D | 3 | 1 | 1 | 1/3 |
| 0.3 | H | 3 | 2 | 1 | 2/3 |
| 0.2 | H | 3 | 3 | 1 | 1 |

The area under the staircase is \(\tfrac{1}{3}\cdot\tfrac{2}{3} + \tfrac{2}{3}\cdot 1 = \tfrac{8}{9} \approx 0.889\). The pairwise interpretation agrees: of the \(3 \times 3 = 9\) (D, H) pairs, the D scored 0.9 and 0.7 each outrank all three H (6 pairs) and the D scored 0.4 outranks two (2 pairs); \(8/9\). The chunk confirms both and checks them against the <code>pROC</code> package,<sup class="c"><a href="#r33">[33]</a></sup> then applies them to the cross-validated probabilities of Section 5.2.



In [ ]:
# ROC curve and AUC by hand (sorting and cumulative counting), checked against pROC.
roc_by_hand <- function(y_true, score) {                             # FPR, TPR arrays and the trapezoidal AUC
  ord <- order(-score); yt <- y_true[ord]
  tpr <- c(0, cumsum(yt == 1)) / sum(yt == 1); fpr <- c(0, cumsum(yt == 0)) / sum(yt == 0)
  list(fpr = fpr, tpr = tpr, auc = sum(diff(fpr) * (head(tpr, -1) + tail(tpr, -1)) / 2))
}
auc_by_pairs <- function(y_true, score) {                            # fraction of (pos, neg) pairs with the positive scored higher
  d <- outer(score[y_true == 1], score[y_true == 0], "-"); (sum(d > 0) + 0.5 * sum(d == 0)) / length(d)
}
toy_y <- c(1, 1, 0, 1, 0, 0); toy_s <- c(0.9, 0.7, 0.6, 0.4, 0.3, 0.2); rt <- roc_by_hand(toy_y, toy_s)
cat("toy example: FPR", round(rt$fpr, 3), "\n             TPR", round(rt$tpr, 3), "\n")
cat(sprintf("toy AUC: trapezoid %.4f | pairwise %.4f | pROC %.4f  (8/9 = %.4f)\n",
            rt$auc, auc_by_pairs(toy_y, toy_s), as.numeric(auc(toy_y, toy_s, quiet = TRUE, direction = "<")), 8/9))
rh <- roc_by_hand(y_pos, prob_cv)
cat(sprintf("breast cancer (CV probabilities): AUC by hand %.4f | pairwise %.4f | pROC %.4f\n",
            rh$auc, auc_by_pairs(y_pos, prob_cv), as.numeric(auc(y_pos, prob_cv, quiet = TRUE))))



The three computations agree on the toy example (8/9) and on the real one, where the AUC of the cross-validated logistic-regression probabilities is about 0.99. The chunk below draws the ROC curve and, next to it, the <strong>precision–recall</strong> (PR) curve for the balanced breast-cancer problem and for the 1 % problem of Section 5.3, because the comparison is the point.



In [ ]:
# ROC and precision-recall curves for a balanced problem (breast cancer) and an imbalanced one (1% prevalence).
pr_by_hand <- function(y_true, score) { ord <- order(-score); yt <- y_true[ord]
  tp <- cumsum(yt == 1); fp <- cumsum(yt == 0); list(recall = tp / sum(yt == 1), precision = tp / (tp + fp)) }
cases <- list("breast cancer (35% malignant)" = list(y_pos, prob_cv, BLUE), "simulated screening (1% positive)" = list(yi_te, prob_imb, ORANGE))
par(mfrow = c(1, 2))
plot(NA, xlim = c(0, 1), ylim = c(0, 1.02), xlab = "false-positive rate (1 - specificity)", ylab = "true-positive rate (sensitivity)", main = "ROC curves (dot = threshold 0.5)")
abline(0, 1, lty = 2); for (nm in names(cases)) { cs <- cases[[nm]]; r <- roc_by_hand(cs[[1]], cs[[2]])
  lines(r$fpr, r$tpr, col = cs[[3]], lwd = 2)
  i0 <- which.min(abs(sort(cs[[2]], decreasing = TRUE) - 0.5)); points(r$fpr[i0 + 1], r$tpr[i0 + 1], pch = 21, bg = cs[[3]], cex = 1.2) }
legend("bottomright", bty = "n", cex = 0.75, lwd = 2, col = c(BLUE, ORANGE, "black"), lty = c(1, 1, 2),
       legend = c(sprintf("breast cancer: AUC = %.3f", roc_by_hand(y_pos, prob_cv)$auc), sprintf("screening 1%%: AUC = %.3f", roc_by_hand(yi_te, prob_imb)$auc), "chance (AUC = 0.5)"))
plot(NA, xlim = c(0, 1), ylim = c(0, 1.02), xlab = "recall (sensitivity)", ylab = "precision (PPV)", main = "Precision-recall curves (dotted = prevalence)")
for (nm in names(cases)) { cs <- cases[[nm]]; pr <- pr_by_hand(cs[[1]], cs[[2]]); lines(pr$recall, pr$precision, col = cs[[3]], lwd = 2); abline(h = mean(cs[[1]]), col = cs[[3]], lty = 3) }
legend("bottomleft", bty = "n", cex = 0.75, lwd = 2, col = c(BLUE, ORANGE), legend = c("breast cancer", "screening 1%"))



<p class="caption"><b>Figure 8.</b> Left: ROC curves of the two problems; both AUCs are high and both curves hug the upper-left corner. Right: PR curves of the same models. For the balanced problem the PR curve is as flattering as the ROC curve; for the 1 % problem it falls away quickly, and its baseline (dotted), the precision of a random classifier, is the prevalence, 0.01. Dots mark the 0.5 threshold.</p>

The ROC curve is invariant to prevalence because both axes are within-class rates — its strength for comparing tests across populations and its weakness in rare-disease problems, where a false-positive rate of 2 % looks negligible but, applied to 9,900 healthy people, produces about 200 false alarms for every 50 detections. The PR curve exposes that, which is why Saito and Rehmsmeier recommend it for imbalanced data;<sup class="c"><a href="#r6">[6]</a></sup> Davis and Goadrich showed that a curve dominates in ROC space if and only if it dominates in PR space, so the two never disagree about <em>which</em> model is better, only about how much better it looks.<sup class="c"><a href="#r30">[30]</a></sup>

### 5.5 Choosing a threshold by cost

A ROC curve is a menu; deploying a classifier means choosing one point on it, and the right point depends on what the errors cost. If a false negative costs \(c_{\mathrm{FN}}\) and a false positive \(c_{\mathrm{FP}}\), a patient with calibrated probability \(p\) should be called positive when the expected cost of calling them negative exceeds that of calling them positive, \(p\,c_{\mathrm{FN}} > (1-p)\,c_{\mathrm{FP}}\), that is, when

<div class="eq">\[ p > t^{*} = \frac{c_{\mathrm{FP}}}{c_{\mathrm{FP}} + c_{\mathrm{FN}}}. \]
<span class="where">If a missed cancer is judged ten times as costly as an unnecessary biopsy, \(t^{*} = 1/11 \approx 0.09\), far below 0.5. Vickers and Elkin's <em>decision curve analysis</em> builds this reasoning into a standard graphical method.<sup class="c"><a href="#r31">[31]</a></sup> The chunk computes the total cost per 100 patients at every threshold and compares the empirical minimum with \(t^{*}\).</span></div>



In [ ]:
# Expected cost per 100 patients as a function of the decision threshold, for three cost ratios.
thresholds <- seq(0.01, 0.99, length.out = 197)
cost_curve <- function(y_true, prob, c_fn, c_fp) sapply(thresholds, function(t) {
  pred <- as.integer(prob >= t); (sum(pred == 0 & y_true == 1) * c_fn + sum(pred == 1 & y_true == 0) * c_fp) / length(y_true) * 100 })
plot(NA, xlim = c(0, 1), ylim = c(0, 20), xlab = "decision threshold on P(malignant)", ylab = "total cost per 100 patients (biopsy = 1)",
     main = "Choosing the threshold by the cost of the two errors (dotted = t*)")
for (cc in list(c(1, 1, GREY), c(10, 1, BLUE), c(30, 1, ORANGE))) {
  c_fn <- as.numeric(cc[1]); c_fp <- as.numeric(cc[2]); col <- cc[3]; curve_v <- cost_curve(y_pos, prob_cv, c_fn, c_fp)
  t_emp <- thresholds[which.min(curve_v)]; t_th <- c_fp / (c_fp + c_fn)
  lines(thresholds, curve_v, col = col, lwd = 2); abline(v = t_th, col = col, lty = 3)
  cat(sprintf("cost ratio %2d:1  cost at t=0.5: %5.1f  at empirical optimum t=%.2f: %5.1f  at t*=%.3f: %5.1f\n",
              c_fn, curve_v[which.min(abs(thresholds - 0.5))], t_emp, min(curve_v), t_th, curve_v[which.min(abs(thresholds - t_th))]))
}
legend("top", bty = "n", cex = 0.78, lwd = 2, col = c(GREY, BLUE, ORANGE), legend = c("missed = 1 x biopsy", "missed = 10 x biopsy", "missed = 30 x biopsy"))



<p class="caption"><b>Figure 9.</b> Total cost per 100 patients against the threshold for three cost ratios, from the cross-validated probabilities. For the asymmetric costs the empirical optimum tracks the theoretical \(t^{*}\) (dotted) closely — about 0.08 against \(t^{*}=0.09\) for a 10:1 ratio, 0.03 against 0.03 for 30:1 — and the cost of staying at 0.5 grows sharply as a missed cancer becomes dearer. With equal costs the curve is shallow and almost flat, so its minimum is poorly determined (here near 0.12, yet barely cheaper than 0.5): when the two errors cost the same, the threshold hardly matters.</p>

### 5.6 Calibration: a probability must mean what it says

The threshold rule assumes that a predicted probability of 0.3 means that about 30 % of such patients have the disease. That property is <strong>calibration</strong>, and it is separate from discrimination: a model can rank patients perfectly (AUC = 1) while its probabilities are all compressed into [0.45, 0.55] or all pushed to 0 and 1. Van Calster and colleagues call calibration "the Achilles heel of predictive analytics" because it is rarely assessed and routinely broken by dataset shift, class re-weighting and over-fitting;<sup class="c"><a href="#r10">[10]</a></sup> Steyerberg and Vergouwe place it, with discrimination and clinical usefulness, among the three things a prediction model must demonstrate.<sup class="c"><a href="#r14">[14]</a></sup>

Two tools assess it. The <strong>reliability diagram</strong> bins cases by predicted probability and plots, per bin, the observed fraction of positives against the mean prediction; a calibrated model lies on the diagonal. The <strong>Brier score</strong> is the mean squared difference between predicted probability and outcome,

<div class="eq">\[ \mathrm{BS} = \frac{1}{n} \sum_{i=1}^{n} (p_i - y_i)^2, \]
<span class="where">introduced by Brier in 1950 for weather forecasts;<sup class="c"><a href="#r9">[9]</a></sup> 0 is perfect, 0.25 is always predicting 0.5, and predicting the prevalence \(\pi\) for everyone scores \(\pi(1-\pi)\). It rewards discrimination and calibration at once. The chunk compares three classifiers of the course, using cross-validated probabilities.</span></div>



In [ ]:
# Reliability diagrams and Brier scores for three classifiers (cross-validated probabilities, positive = malignant).
cvprob_knn <- function(Xm, y, k = 15, folds = 10, seed = 0) { f <- strat_folds(y, folds, seed); p <- numeric(length(y))
  for (j in 1:folds) { tr <- f != j; pr <- knn(Xm[tr, ], Xm[!tr, ], factor(y[tr]), k = k, prob = TRUE)
    w <- attr(pr, "prob"); p[!tr] <- ifelse(pr == 1, w, 1 - w) }; p }
cvprob_nb <- function(Xm, y, folds = 10, seed = 0) { f <- strat_folds(y, folds, seed); p <- numeric(length(y)); df <- data.frame(Xm, yf = factor(y))
  for (j in 1:folds) { tr <- f != j; m <- naiveBayes(yf ~ ., df[tr, ]); p[!tr] <- predict(m, df[!tr, ], type = "raw")[, 2] }; p }
probs <- list("logistic regression" = prob_cv, "15-nearest neighbours" = cvprob_knn(Xs, y_pos), "Gaussian naive Bayes (Unit 3)" = cvprob_nb(Xs, y_pos))
reldiag <- function(p, y, nb = 8) { br <- unique(quantile(p, seq(0, 1, length.out = nb + 1))); br[1] <- -Inf; br[length(br)] <- Inf
  b <- cut(p, br, labels = FALSE); data.frame(mp = tapply(p, b, mean), fp = tapply(y, b, mean)) }
plot(NA, xlim = c(0, 1), ylim = c(0, 1), xlab = "mean predicted probability of malignancy (8 quantile bins)", ylab = "observed fraction malignant", main = "Reliability diagram")
abline(0, 1, lty = 2); cols <- c(BLUE, GREEN, ORANGE); rows <- list()
for (i in seq_along(probs)) { rd <- reldiag(probs[[i]], y_pos); lines(rd$mp, rd$fp, type = "o", pch = 19, col = cols[i])
  rows[[names(probs)[i]]] <- round(c(`Brier score` = mean((probs[[i]] - y_pos)^2), AUC = as.numeric(auc(y_pos, probs[[i]], quiet = TRUE)),
    `mean pred. P` = mean(probs[[i]]), `share P outside (.02,.98)` = mean(probs[[i]] < 0.02 | probs[[i]] > 0.98)), 3) }
legend("topleft", bty = "n", cex = 0.85, lwd = 2, pch = 19, col = cols, legend = names(probs))
as.data.frame(do.call(rbind, rows))



<p class="caption"><b>Figure 10.</b> Reliability diagram of three classifiers. Logistic regression, fitted by maximising the likelihood of the labels, follows the diagonal closely. Naive Bayes has a comparable AUC but extreme probabilities — most below 0.02 or above 0.98 — because its independence assumption multiplies evidence that is in fact redundant; it is <em>over-confident</em>, and its Brier score is markedly worse. The 15-NN vote fractions lie in between.</p>

The table separates the two qualities: the AUCs are within a few hundredths of one another, the Brier scores are not. Clinically, over-confidence is not cosmetic: a model that says "2 % risk" when the true risk among such patients is 15 % will be used to withhold a warranted biopsy. Models whose probabilities are not calibrated by construction — naive Bayes, SVMs, most networks — are <em>recalibrated</em> on validation data (Platt scaling, isotonic regression), and the observed-to-expected ratio (mean outcome over mean predicted probability, 1.0 when calibrated-in-the-large) is reported alongside the AUC.

<div class="box"><h4>Exercise 4</h4><p>Recalibrate the naive-Bayes probabilities by fitting an isotonic or logistic recalibration (<code>glm(y ~ p, family = binomial)</code> on out-of-fold probabilities, the R form of Platt scaling) and add the recalibrated curve. It should move onto the diagonal and its Brier score fall, while its AUC barely changes. Why can recalibration improve the Brier score but not the AUC? (Which operation is the ROC curve invariant to?)</p></div>

## 6. Metrics for regression

When the target is a number — disease progression, blood glucose, length of stay, a house price — the errors are residuals \(e_i = y_i - \hat{y}_i\) and the standard summaries are

<div class="eq">\[ \mathrm{MSE} = \frac{1}{n}\sum e_i^2, \qquad \mathrm{RMSE} = \sqrt{\mathrm{MSE}}, \qquad \mathrm{MAE} = \frac{1}{n}\sum |e_i|, \qquad R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}. \]
<span class="where">MSE is what least squares minimises (Unit 2) and what Section 4.3 decomposes; RMSE restores the units of the target; MAE, in the same units, is less sensitive to a few large errors and is the one a reader can interpret directly. \(R^2\) compares the model's squared error with that of the simplest predictor, the mean of \(y\): 1 for a perfect fit, 0 for a model no better than the mean.</span></div>

On the training sample, \(R^2\) of a least-squares fit lies in \([0, 1]\), because the fit can always do at least as well as the mean. On a <em>test</em> sample nothing guarantees that: a model that predicts worse than the test mean has a <strong>negative \(R^2\)</strong>. This is not a bug; it is the correct report that the model is worse than useless on those data. The Python edition used the diabetes-progression data of Efron and colleagues;<sup class="c"><a href="#r21">[21]</a></sup> the R edition uses the Boston housing data of Harrison and Rubinfeld (506 towns, 13 predictors, median home value as the target),<sup class="c"><a href="#r36">[36]</a></sup> which the <code>mlbench</code> package supplies. The chunks show the healthy case and the pathological one.



In [ ]:
# Regression metrics by hand (from residuals) and the training R^2 that R's summary() reports, on the Boston data.
data(BostonHousing, package = "mlbench"); bh <- BostonHousing; bh$chas <- as.numeric(as.character(bh$chas))
set.seed(0); bi <- sample(nrow(bh), round(0.7 * nrow(bh)))
lin <- lm(medv ~ ., bh[bi, ]); yd_te <- bh$medv[-bi]; pe <- predict(lin, bh[-bi, ]); e <- yd_te - pe
by_hand <- c(MSE = mean(e^2), RMSE = sqrt(mean(e^2)), MAE = mean(abs(e)), `test R2` = 1 - sum(e^2) / sum((yd_te - mean(yd_te))^2))
cat(sprintf("target medv: mean %.1f, SD %.1f on the %d test towns | training R2 (summary) = %.3f\n",
            mean(yd_te), sd(yd_te), length(yd_te), summary(lin)$r.squared))
round(by_hand, 3)



The RMSE exceeds the MAE, as it always does when a few residuals are large; both should be read against the spread of the target (SD about 9): the model removes most, not all, of the variation, which the test \(R^2\) says in relative terms. Here the training \(R^2\) (about 0.74) and the test \(R^2\) (about 0.74) are close, because linear regression is a low-variance model whose optimism is small; Section 1.2's gap appears only with a more flexible fit, which the next chunk supplies. Now the pathological case.



In [ ]:
# Negative test R^2: multivariate polynomial regression of increasing degree on the same split.
preds <- setdiff(names(bh), "medv"); rows <- list()
for (dg in 1:3) {
  form <- as.formula(paste0("medv ~ polym(", paste(preds, collapse = ", "), ", degree = ", dg, ", raw = TRUE)"))
  fit <- suppressWarnings(lm(form, bh[bi, ])); e <- yd_te - predict(fit, bh[-bi, ])
  rows[[dg]] <- data.frame(degree = dg, `no. coefficients` = length(coef(fit)), `train R2` = summary(fit)$r.squared,
                           `test R2` = 1 - sum(e^2, na.rm = TRUE) / sum((yd_te - mean(yd_te))^2),
                           `test RMSE` = sqrt(mean(e^2, na.rm = TRUE)), check.names = FALSE)
}
rmse_mean <- sqrt(mean((yd_te - mean(bh$medv[bi]))^2))
cat(sprintf("RMSE of predicting the training mean for every test town: %.1f  (R2 = %.4f)\n",
            rmse_mean, 1 - sum((yd_te - mean(bh$medv[bi]))^2) / sum((yd_te - mean(yd_te))^2)))
res <- do.call(rbind, rows); rownames(res) <- NULL; res[, 3:5] <- signif(res[, 3:5], 3); res



A degree-3 multivariate polynomial has hundreds of coefficients fitted to about 350 towns. Its training \(R^2\) climbs to 1.000 and its test \(R^2\) is a large negative number: its predictions are far worse than telling every town the average value. This is Section 4 in regression clothing. Note also that the trivial mean predictor scores an \(R^2\) of essentially zero on the test set (it differs from exactly zero only because the training mean is not the test mean).

<div class="box key"><h4>Reporting a regression model</h4><p>Report RMSE or MAE in the units of the target, with the SD of the target beside it, and the test \(R^2\). Never \(R^2\) alone: 0.9 for a quantity that varies a thousand-fold may hide clinically useless errors, and 0.3 may be excellent for a noisy outcome.</p></div>

## 7. Data leakage

<strong>Leakage</strong> is the use, during training, of information that will not be available when the model is deployed — information that leaks from the test set, or from the future, into the model. It produces evaluations that are honest in form (a test set was held out, cross-validation was run) and wrong in substance. Kaufman and colleagues gave the first systematic account;<sup class="c"><a href="#r11">[11]</a></sup> Kapoor and Narayanan surveyed 294 papers across 17 fields whose results were later found inflated by leakage, and proposed a taxonomy and checklist.<sup class="c"><a href="#r12">[12]</a></sup> This section demonstrates the three forms that account for most cases, each with the inflated number beside the honest one.

### 7.1 Preprocessing before splitting

Any step that <em>learns from the data</em> — feature selection, standardisation, imputation, dimensionality reduction — must be fitted on the training fold only. Selecting features on the full dataset and then cross-validating the classifier is the most damaging version, because the selection has already seen the test labels; Ambroise and McLachlan showed in 2002 that gene-expression classifiers evaluated this way reported near-zero error on data with no signal.<sup class="c"><a href="#r24">[24]</a></sup> The remedy is to put every learned step <em>inside</em> the cross-validation loop, refitted on each training fold. The chunk shows the leak on pure noise (1,000 features, 200 cases, random labels) and, for contrast, the scaling leak on the breast-cancer data.



In [ ]:
# Leak 1: feature selection (and, for contrast, scaling) performed before splitting versus inside the fold loop.
select_k <- function(Xm, y, k) order(-abs(apply(Xm, 2, function(col) cor(col, y))))[1:k]   # top-k by |correlation| with the label
set.seed(0); leaky <- honest <- numeric(5)
for (rep in 1:5) {                                                   # 5 independent noise datasets
  Xn <- matrix(rnorm(200 * 1000), 200, 1000); yn <- sample(0:1, 200, replace = TRUE); f <- strat_folds(yn, 5, 0)
  sel <- select_k(Xn, yn, 20); Xsel <- Xn[, sel]                     # WRONG: selection has seen every label, test folds included
  leaky[rep] <- mean(sapply(1:5, function(j) { tr <- f != j; mean(knn(Xsel[tr, ], Xsel[!tr, ], factor(yn[tr]), 5) == yn[!tr]) }))
  honest[rep] <- mean(sapply(1:5, function(j) { tr <- f != j; s <- select_k(Xn[tr, ], yn[tr], 20)   # selection inside the fold
    mean(knn(Xn[tr, s], Xn[!tr, s], factor(yn[tr]), 5) == yn[!tr]) }))
}
cat(sprintf("feature selection on NOISE (true accuracy 0.5): selected before CV %.3f | selected inside the loop %.3f\n", mean(leaky), mean(honest)))
cv5 <- strat_kfold(y_pos, 5, 0)
leaky_sc <- mean(sapply(cv5, function(f) mean(knn(scale(X)[f$train, ], scale(X)[f$test, ], factor(y_pos[f$train]), 5) == y_pos[f$test])))   # scaled on all data
honest_sc <- mean(sapply(cv5, function(f) { sc <- scale(X[f$train, ]); te <- scale(X[f$test, ], attr(sc, "scaled:center"), attr(sc, "scaled:scale"))
  mean(knn(sc, te, factor(y_pos[f$train]), 5) == y_pos[f$test]) }))
cat(sprintf("scaling on the breast-cancer data: scaled before CV %.4f | scaled inside the fold %.4f\n", leaky_sc, honest_sc))



On data with no signal, selecting the 20 "best" of 1,000 noise features before cross-validation yields an accuracy well above chance — a result that would pass review — while the same selection inside the loop gives the honest value near 0.5. The scaling leak changes the breast-cancer accuracy by almost nothing: the scaler learns two numbers per feature, which the test fold barely moves. The damage is proportional to how much the preprocessing learns from the labels; refitting inside the loop costs nothing and covers both.

### 7.2 Target-derived features

The second leak is a feature that is a consequence of the outcome rather than a predictor of it: the treatment prescribed after the diagnosis, a test ordered because the clinician already suspected the disease, a billing code generated by the event being predicted. Such features are extremely predictive in the retrospective database and unavailable at prediction time. The simulation adds a "post-biopsy treatment code" that agrees with the biopsy result in 95 % of cases.



In [ ]:
# Leak 2: a feature derived from the outcome (available in the database, not at prediction time).
set.seed(1); treatment_code <- ifelse(runif(nrow(X)) < 0.95, y_pos, 1 - y_pos)   # 'treatment given' agrees with biopsy 95% of the time
Xleak <- cbind(Xs, treatment = treatment_code)
cv_acc <- function(Xm) mean(sapply(strat_kfold(y_pos, 10, 0), function(f) {
  fit <- suppressWarnings(glm(y ~ ., data.frame(Xm, y = y_pos)[f$train, ], family = binomial))
  mean((predict(fit, data.frame(Xm, y = y_pos)[f$test, ], type = "response") >= 0.5) == y_pos[f$test]) }))
honest_acc <- cv_acc(Xs); leaky_acc <- cv_acc(Xleak)
coefs <- coef(suppressWarnings(glm(y ~ ., data.frame(Xleak, y = y_pos), family = binomial)))[-1]
rank <- which(order(-abs(coefs)) == length(coefs))
cat(sprintf("CV accuracy without the leaked feature: %.3f   with it: %.3f\n", honest_acc, leaky_acc))
cat(sprintf("the leaked feature has the largest-ranked standardised coefficient #%d of %d (|coef| = %.2f, next largest %.2f)\n",
            rank, length(coefs), abs(coefs[length(coefs)]), sort(abs(coefs), decreasing = TRUE)[2]))



The accuracy rises and the model leans on the leaked column more than on any nuclear measurement. Cross-validation cannot detect this, because the leak is present in every fold. The defence is clinical, not statistical — for every feature, ask <em>is this value known at the moment the prediction is needed?</em> — and the TRIPOD+AI item on the timing of each predictor exists for this reason (Section 8).

### 7.3 Repeated measurements per patient

The third leak is structural. When patients contribute several records — visits, images, ECG segments — random splitting puts some records of a patient in training and others in test. The model then recognises the <em>patient</em>, not the condition. Saeb and colleagues showed that record-wise cross-validation of clinical sensor data inflates accuracy dramatically relative to subject-wise cross-validation, and that most published studies had used the wrong one.<sup class="c"><a href="#r25">[25]</a></sup> The remedy is a grouped split (Figure 3). The simulation builds 100 patients with 8 records each; each patient has an individual baseline on all features (which makes their records recognisable) and the disease shifts three features moderately.



In [ ]:
# Leak 3: several records per patient, split at random (record-wise) versus by patient (grouped).
set.seed(0); n_pat <- 100; n_rec <- 8
label_pat <- sample(0:1, n_pat, replace = TRUE)                       # one diagnosis per patient
baseline_pat <- matrix(rnorm(n_pat * 10, 0, 1.5), n_pat, 10)          # patient-specific baseline on 10 features
effect <- 1.5 * c(rep(1, 3), rep(0, 7))                               # disease shifts 3 of the 10 features
X_rep <- do.call(rbind, lapply(1:n_pat, function(p) t(sapply(1:n_rec, function(i) baseline_pat[p, ] + effect * label_pat[p] + rnorm(10)))))
y_rep <- rep(label_pat, each = n_rec); patient <- rep(1:n_pat, each = n_rec)
sc_cv <- function(folds) mean(sapply(folds, function(f) { sc <- scale(X_rep[f$train, ]); te <- scale(X_rep[f$test, ], attr(sc, "scaled:center"), attr(sc, "scaled:scale"))
  mean(knn(sc, te, factor(y_rep[f$train]), 5) == y_rep[f$test]) }))
record_folds <- strat_kfold(y_rep, 5, 0)
set.seed(0); pat_fold <- sample(rep(1:5, length.out = n_pat))[patient]
group_folds <- lapply(1:5, function(j) list(train = which(pat_fold != j), test = which(pat_fold == j)))
cat(sprintf("%d records from %d patients\n", length(y_rep), n_pat))
cat(sprintf("record-wise 5-fold CV (patients leak across folds): %.3f\n", sc_cv(record_folds)))
cat(sprintf("patient-wise 5-fold CV (grouped):                   %.3f\n", sc_cv(group_folds)))



Record-wise cross-validation reports an accuracy far above the patient-wise one. The first answers "can the model classify a new record from a patient it has already seen?" — almost never the clinical question. The second answers "can it classify a new patient?", and is much lower because the disease effect is modest relative to between-patient variation. The same logic applies to any grouping that will not recur at deployment: split by hospital to estimate performance at a new hospital, by year to estimate performance next year.

<div class="box warn"><h4>Leakage is silent</h4><p>None of the three leaks produces a warning or an implausible number — the leaked accuracies above are values a reviewer would accept. They are found by reasoning about the data-generating process, not by inspecting output. Kapoor and Narayanan's checklist — clean train–test separation at every preprocessing step? any feature unavailable at prediction time? duplicates or groups across the split? test distribution matching deployment? — should be run on every evaluation before its number is believed.<sup class="c"><a href="#r12">[12]</a></sup></p></div>

## 8. Internal versus external validation, and reporting

Everything so far is <strong>internal validation</strong>: the test cases, however carefully held out, come from the same source as the training cases — same hospital, scanners, years, inclusion criteria, labelling. It estimates performance <em>on that source</em>. Performance elsewhere is a different quantity, estimated only by <strong>external validation</strong>: applying the frozen model, without refitting, to data collected elsewhere, later, or under different criteria.<sup class="c"><a href="#r14">[14]</a>,<a href="#r28">[28]</a></sup> It reveals what internal validation cannot: shortcuts (Unit 1, where a scanner identifier predicted the label in one hospital and not elsewhere), miscalibration from a different prevalence or case mix, and deterioration from population differences. The sepsis model of Unit 1 is the canonical case: excellent internal numbers, a sensitivity of 33 % at external validation.<sup class="c"><a href="#r32">[32]</a></sup>

The chunk simulates the structure with the breast-cancer data, using one feature as a stand-in for "site": the model is developed on tumours with below-median <em>mitotic count</em> (one "hospital", where early-stage tumours dominate) and applied, frozen, to those above the median (another "hospital" that sees mostly advanced, more often malignant tumours).



In [ ]:
# Internal (cross-validated, same 'hospital') versus external (frozen model, other 'hospital') validation.
site <- X[, "Mitoses"] > median(X[, "Mitoses"])                     # FALSE = development site, TRUE = external site
dev <- !site; ext <- site
prob_int <- cvprob_glm(as.matrix(Xs[dev, ]), y_pos[dev])            # internal: 10-fold CV on the development site
fit_dev <- suppressWarnings(glm(y ~ ., data.frame(Xs[dev, ], y = y_pos[dev]), family = binomial))
prob_ext <- predict(fit_dev, data.frame(Xs[ext, ]), type = "response")   # frozen model, never refitted
report <- function(yt, pr) { pred <- as.integer(pr >= 0.5)
  tp <- sum(pred == 1 & yt == 1); fn <- sum(pred == 0 & yt == 1); fp <- sum(pred == 1 & yt == 0); tn <- sum(pred == 0 & yt == 0)
  round(c(prevalence = mean(yt), accuracy = (tp + tn) / length(yt), sensitivity = tp / (tp + fn), specificity = tn / (tn + fp),
          AUC = as.numeric(auc(yt, pr, quiet = TRUE)), Brier = mean((pr - yt)^2), `obs/exp` = mean(yt) / mean(pr)), 3) }
cat(sprintf("development site: %d tumours | external site: %d tumours\n", sum(dev), sum(ext)))
as.data.frame(rbind(`internal (CV, development site)` = report(y_pos[dev], prob_int), `external (frozen model, other site)` = report(y_pos[ext], prob_ext)))



The pattern is the one Van Calster warns about.<sup class="c"><a href="#r10">[10]</a></sup> Discrimination survives: the external AUC stays high (about 0.95 against the internal 0.99), so the model still ranks malignant above benign, and calibration-in-the-large holds too — the observed/expected ratio stays near 1. What does not survive is the operating point of the frozen threshold. At the external site, where the tumours have high mitotic counts and malignancy is far more common, the same 0.5 cut-off sends many more benign tumours to biopsy: specificity falls from about 0.98 to 0.85 while sensitivity rises, and the Brier score worsens relative to the internal estimate; accuracy barely moves, only because the external set is now about 89 % malignant, so it has stopped being an informative summary. A report giving only the AUC would call this model transportable; one giving the threshold-dependent metrics at the deployed prevalence would call for the threshold to be re-derived before deployment — the correct conclusion. What is general is that a frozen threshold does not survive a change of case mix, even when discrimination and calibration-in-the-large do.

### 8.1 What to report: TRIPOD+AI

The <strong>TRIPOD+AI</strong> statement (Transparent Reporting of a multivariable prediction model for Individual Prognosis Or Diagnosis, 2024 update for machine learning) is the reporting standard medical journals require for prediction models.<sup class="c"><a href="#r13">[13]</a></sup> In the vocabulary of this supplement, its items that concern evaluation are:

- **Source of data and setting** for development and each validation; dates; eligibility criteria — so the reader can judge what the test sample is a sample <em>of</em>.
- **Outcome** definition and timing; **predictors**, with the time at which each is measured relative to the outcome (the defence against Section 7.2).
- **Sample size** and its justification; Riley and colleagues give the calculation,<sup class="c"><a href="#r27">[27]</a></sup> and Section 2.1's binomial formula is the back-of-the-envelope version.
- **Model development**: all candidate models and hyperparameters tried, and how they were selected (Section 3.4). **Missing data** handled inside the loop.
- **Validation method**: internal (which resampling, how many folds, nested or not, grouped by what) and external (which population).
- **Performance measures**: discrimination (AUC with a confidence interval), calibration (plot and summary), threshold-dependent measures at a <em>pre-specified</em> threshold with the prevalence of the sample (Sections 5.3–5.6); for regression, error in the units of the target (Section 6). Steyerberg's framework organises these.<sup class="c"><a href="#r26">[26]</a></sup>
- **Fairness**: performance by relevant subgroups (sex, age, ethnicity, site), the TRIPOD+AI addition that connects to Unit 1. **Availability** of code, data and model.

Re-read the shortcut example of Unit 1 with this list. The model with the scanner feature reported a training accuracy (Section 1.2: not a measurement), an accuracy on new cases from the same hospital (internal validation) and an accuracy at a new hospital (external validation) — and only the third revealed the failure. A TRIPOD+AI-compliant report would also have listed "scanner identifier" among the predictors with its timing, and a reader would have asked why a scanner should predict a diagnosis.

<div class="box bridge"><h4>Bridge to the course workshops</h4><p>Workshop 2 asks you to present an article that applies supervised learning. Use the list above as the reading grid: for each item, note what the article reports and what it omits. An article that reports accuracy without the test size, a threshold-dependent metric without the prevalence, or an AUC without calibration has not provided the evidence a clinician needs; saying so precisely is the skill the workshop assesses.</p></div>

## 9. A reusable evaluation function

The later units need, repeatedly, the same evaluation: repeated stratified cross-validation of several classifiers with the metrics of Section 5, AUC and Brier score, and a measure of uncertainty. The function below packages it; copy it into any notebook. It takes a list of model wrappers (each returning out-of-fold probabilities of the positive class), data with the positive class coded 1, and a fold assignment (default 5-fold stratified repeated 5 times), and returns mean and SD across folds per metric. All models are compared on the <em>same</em> folds, which removes the split-to-split noise of Figure 2 from the comparison.



In [ ]:
# A reusable evaluation function for binary classifiers (copy this chunk into later notebooks).
evaluate_classifiers <- function(models, Xm, y, k = 5, repeats = 5, threshold = 0.5) {
  # models: named list of functions (Xtr, ytr, Xte) -> P(positive) on Xte;  y: 0/1 with 1 = condition to detect.
  recs <- list()
  for (rp in 1:repeats) { folds <- strat_kfold(y, k, seed = rp)
    for (nm in names(models)) for (f in folds) {
      prob <- models[[nm]](Xm[f$train, , drop = FALSE], y[f$train], Xm[f$test, , drop = FALSE])
      pred <- as.integer(prob >= threshold); yt <- y[f$test]
      tp <- sum(pred == 1 & yt == 1); fn <- sum(pred == 0 & yt == 1); fp <- sum(pred == 1 & yt == 0); tn <- sum(pred == 0 & yt == 0)
      m <- metrics_from_counts(tp, fn, fp, tn)
      recs[[length(recs) + 1]] <- data.frame(model = nm, accuracy = unname(m["accuracy"]), sensitivity = unname(m["sensitivity (recall)"]),
        specificity = unname(m["specificity"]), AUC = as.numeric(auc(yt, prob, quiet = TRUE)), Brier = mean((prob - yt)^2)) }
  }
  long <- do.call(rbind, recs)
  agg <- aggregate(cbind(accuracy, sensitivity, specificity, AUC, Brier) ~ model, long, function(z) c(mean = mean(z), sd = sd(z)))
  do.call(data.frame, agg)
}
m_glm <- function(Xtr, ytr, Xte) { fit <- suppressWarnings(glm(y ~ ., data.frame(Xtr, y = ytr), family = binomial)); predict(fit, data.frame(Xte), type = "response") }
m_knn <- function(Xtr, ytr, Xte) { pr <- knn(Xtr, Xte, factor(ytr), k = 5, prob = TRUE); w <- attr(pr, "prob"); ifelse(pr == 1, w, 1 - w) }
m_nb  <- function(Xtr, ytr, Xte) predict(naiveBayes(yf ~ ., data.frame(Xtr, yf = factor(ytr))), data.frame(Xte), type = "raw")[, 2]
results <- evaluate_classifiers(list("logistic regression" = m_glm, "5-nearest neighbours" = m_knn, "Gaussian naive Bayes" = m_nb), Xs, y_pos)
round_df <- results; num <- sapply(round_df, is.numeric); round_df[num] <- lapply(round_df[num], round, 3); round_df



The table reads as a TRIPOD-style summary: discrimination (AUC), calibration (Brier) and the two clinical rates at the pre-specified threshold, each with its fold-to-fold SD, on identical folds. Differences smaller than about one fold-to-fold SD should not be over-interpreted; the Brier difference between naive Bayes (about 0.037) and the others (about 0.025) is larger than that scatter and, averaged over the 25 folds, is a real difference in calibration. Units 3, 4 and 6 call a function of this shape with their own model lists.

<div class="box"><h4>Exercise 5</h4><p>Call the function with the grouped folds and the repeated-measurement data of Section 7.3 (replace <code>strat_kfold</code> inside by the grouped assignment), and with <code>threshold = 0.09</code> on the breast-cancer data (the cost-derived threshold of Section 5.5). Explain every column that changes between the 0.5 and the 0.09 threshold, and the one column that does not.</p></div>

## 10. Summary

You should now be able to explain, without notes:

- Why the training error is optimistically biased, and the roles of the training, validation and test samples (and that the test sample is used once).
- Why a single hold-out accuracy is one draw from a distribution whose width is set by the test size, and what stratification protects against.
- How k-fold cross-validation works, by hand; when stratified, repeated, leave-one-out and grouped variants are appropriate; and why selecting hyperparameters and reporting performance on the same folds is optimistic (nested CV).
- Overfitting and underfitting in terms of model complexity; the bias–variance decomposition, its U-shaped total, and how the complexity curve and the learning curve display it; the curse of dimensionality and the \(p/n\) ratio.
- The confusion matrix and every metric derived from it, with both the machine-learning and the clinical names; why accuracy fails under imbalance; why PPV depends on prevalence (Bayes) while sensitivity and specificity do not.
- How a ROC curve is built from scores, what the AUC means as a probability, when the precision–recall curve is more informative, how a threshold follows from the costs of the two errors, and what calibration is and how to assess it.
- MSE, RMSE, MAE and \(R^2\), and why \(R^2\) is negative when a model is worse than the mean.
- The three classic leaks — preprocessing before splitting, target-derived features, repeated patients across folds — how each inflates a number, and how each is prevented.
- The difference between internal and external validation, what each estimates, and the TRIPOD+AI items a model report must contain.

## 11. Self-assessment

<details><summary>1. A colleague reports "our neural network reaches 98 % accuracy on the training set". List the three questions you would ask before taking the number seriously, and say what a 1-NN classifier teaches about the first of them.</summary>On which cases was it measured (training, validation or test); how were they separated from the training data (random split, by patient, by hospital, by date); how many were there. On the training set, 1-NN reaches essentially 100 % by construction whatever the data, so training accuracy is not a measurement of anything.</details>

<details><summary>2. A test set has 50 cases and the measured accuracy is 0.90. Give the approximate standard error and say whether a second model with 0.94 on the same test set has been shown to be better.</summary>\(\sqrt{0.9 \times 0.1 / 50} \approx 0.042\). A difference of 0.04 is about one standard error; it has not been shown. Paired comparison on the same folds, or a much larger test set, is needed.</details>

<details><summary>3. Write down the test indices of the four folds of unshuffled k-fold on 10 cases (indices 1–10), with nearly equal blocks.</summary>{1, 2, 3}, {4, 5, 6}, {7, 8}, {9, 10} (or a similar near-equal partition). The training set of each fold is the complement. If the cases were sorted by class, the first fold would train on a sample depleted of its own class, which is why a shuffle is used.</details>

<details><summary>4. Explain in one sentence each why leave-one-out has low bias and high variance, and when stratified k-fold is preferred over plain k-fold.</summary>Low bias because each model is trained on \(n-1\) cases, almost the full sample; high variance because the \(n\) training sets overlap almost completely, so the \(n\) errors are highly correlated and their average fluctuates with the particular sample. Stratified k-fold is preferred when the dataset is small or imbalanced, so that each fold contains the same proportion of each class.</details>

<details><summary>5. In the nested-CV experiment on noise, the best inner score was above 0.5 and the nested estimate was about 0.5. Which number would appear in a paper that tuned on the test set, and what general principle does the gap illustrate?</summary>The best inner score — the maximum of many noisy estimates, biased upwards. The principle: data used to make a choice (model, hyperparameter, feature, threshold) cannot also be used to report the performance of what was chosen; one more level of holding out is required for every level of looking.</details>

<details><summary>6. Define bias and variance, say which leads to underfitting and which to overfitting, and name the knob that moves k-NN between the two.</summary>Bias is error from erroneous assumptions in the model (too simple a form) and leads to underfitting; variance is sensitivity to small fluctuations of the training set and leads to overfitting. The knob is \(k\): small \(k\) gives low bias and high variance, large \(k\) the reverse.</details>

<details><summary>7. Compute by hand, for TP = 40, FN = 10, FP = 20, TN = 930: accuracy, sensitivity, specificity, PPV, NPV, F1 and balanced accuracy.</summary>Accuracy 970/1000 = 0.970; sensitivity 40/50 = 0.80; specificity 930/950 ≈ 0.979; PPV 40/60 ≈ 0.667; NPV 930/940 ≈ 0.989; F1 = 80/(80 + 20 + 10) ≈ 0.727; balanced accuracy ≈ 0.889. The accuracy is dominated by the 950 negatives.</details>

<details><summary>8. A test has sensitivity 0.95 and specificity 0.95. Compute its PPV at 50 % prevalence and at 1 % prevalence, and explain in one sentence why they differ.</summary>At 50 %: \(0.95 \times 0.5 / (0.95 \times 0.5 + 0.05 \times 0.5) = 0.95\). At 1 %: \(0.0095 / (0.0095 + 0.0495) \approx 0.161\). At low prevalence the false alarms, which come from the large healthy group, outnumber the true positives, which come from the small diseased group.</details>

<details><summary>9. Scores for 4 diseased patients are 0.8, 0.6, 0.5, 0.2 and for 3 healthy patients 0.7, 0.4, 0.1. Compute the AUC by counting pairs.</summary>12 pairs. 0.8 beats 3; 0.6 beats 2 (0.4, 0.1); 0.5 beats 2; 0.2 beats 1 (0.1). AUC = 8/12 ≈ 0.667.</details>

<details><summary>10. Two models have the same AUC of 0.90. One has a Brier score of 0.08 and the other 0.20. What do you conclude, and which of the two can be used with the cost-based threshold rule \(t^{*} = c_{FP}/(c_{FP} + c_{FN})\)?</summary>They rank patients equally well, but the second is poorly calibrated — its probabilities do not match observed frequencies. The threshold rule assumes calibrated probabilities, so only the first can use it directly; the second must be recalibrated (Platt scaling, isotonic regression) first.</details>

<details><summary>11. Read this code and say what is wrong with it and what number it will inflate:<br><code>sel <- select_k(X, y, 10)</code><br><code>acc <- cv_svm(X[, sel], y, folds = 5)</code></summary>Feature selection was fitted on all the data, including the cases that each fold later uses for testing, so the selected features have already "seen" the test labels. The CV accuracy is inflated — on pure noise it rises well above 0.5. The selection must be repeated inside each training fold.</details>

<details><summary>12. A model developed in a tertiary hospital has an external AUC equal to its internal AUC, but its sensitivity at the pre-specified threshold drops from 0.95 to 0.80 and its observed/expected ratio is 1.8. What happened, and what should be done before deployment?</summary>Discrimination is preserved but calibration is not: the external population has a different prevalence or case mix, so the model's probabilities are miscalibrated and fewer cases pass the threshold. The model should be recalibrated on local data (at least the intercept, calibration-in-the-large) and the threshold re-derived from local costs before deployment, and the report must present both validations.</details>

## Glossary

<dl class="gloss">
<dt>AUC (area under the ROC curve)</dt><dd>Probability that a randomly chosen positive case receives a higher score than a randomly chosen negative case; 1 = perfect ranking, 0.5 = random.</dd>
<dt>Balanced accuracy</dt><dd>Mean of sensitivity and specificity; the accuracy the classifier would have if both classes were equally frequent.</dd>
<dt>Bias / variance</dt><dd>Systematic error of the average fit due to erroneous model assumptions (leads to underfitting) / sensitivity of the fit to fluctuations of the training sample (leads to overfitting). Expected squared error = bias² + variance + noise.</dd>
<dt>Brier score</dt><dd>Mean squared difference between predicted probability and observed outcome; measures calibration and discrimination together.</dd>
<dt>Calibration</dt><dd>Agreement between predicted probabilities and observed frequencies; assessed with a reliability diagram and the observed/expected ratio.</dd>
<dt>Confusion matrix</dt><dd>Cross-tabulation of true against predicted class: TP, FN, FP, TN.</dd>
<dt>Cross-validation (k-fold)</dt><dd>Partition of the sample into k folds; each fold is tested once by a model trained on the other k−1; the k scores are averaged. Variants: stratified, repeated, leave-one-out (k = n), grouped.</dd>
<dt>Curse of dimensionality</dt><dd>The sparsity of a fixed sample in a feature space whose dimension grows; distances lose meaning and the parameters-to-observations ratio p/n rises.</dd>
<dt>Data leakage</dt><dd>Use during training of information unavailable at prediction time (test-set statistics, outcome-derived features, records of test patients), inflating the evaluation.</dd>
<dt>External validation</dt><dd>Evaluation of a frozen model on data from a different source (site, period, population) than the development data.</dd>
<dt>F1 score</dt><dd>Harmonic mean of precision and recall with equal weight; ignores true negatives.</dd>
<dt>Generalisation error</dt><dd>Expected loss of a model on new cases from the population; the quantity every evaluation method estimates.</dd>
<dt>Hold-out</dt><dd>A single random split into training and test parts; its estimate has a variance set by the test size.</dd>
<dt>Hyperparameter</dt><dd>A setting of the learning algorithm not learned from the training data (k in k-NN, tree depth, regularisation strength), chosen with validation data or cross-validation.</dd>
<dt>Imbalance</dt><dd>Unequal class frequencies; mild (minority 20–40 %), moderate (1–20 %), extreme (< 1 %). Accuracy becomes uninformative.</dd>
<dt>Matthews correlation coefficient (MCC)</dt><dd>Correlation between true and predicted classes computed from all four cells of the confusion matrix; −1 to +1, 0 at chance.</dd>
<dt>Nested cross-validation</dt><dd>An outer CV loop for estimation around an inner CV loop for hyperparameter selection, so that no test fold is used for any choice.</dd>
<dt>Overfitting / underfitting</dt><dd>Fitting the accidents of the training sample through excess complexity (training score high, test score low) / failing to fit the regularity through insufficient complexity (both low).</dd>
<dt>Precision (PPV) / NPV</dt><dd>Fraction of predicted positives that are truly positive / of predicted negatives that are truly negative; both depend on prevalence.</dd>
<dt>Precision–recall curve</dt><dd>Precision against recall over all thresholds; baseline equals the prevalence; preferred for imbalanced problems. Summarised by average precision.</dd>
<dt>R²</dt><dd>One minus the ratio of the model's squared error to that of predicting the mean; negative on a test set when the model is worse than the mean.</dd>
<dt>ROC curve</dt><dd>True-positive rate against false-positive rate over all thresholds; invariant to prevalence.</dd>
<dt>Sensitivity (recall, TPR) / specificity (TNR)</dt><dd>Fraction of true positives detected / of true negatives cleared; properties of the test, independent of prevalence.</dd>
<dt>Stratification</dt><dd>Splitting within each class so that every part has the same class proportions.</dd>
<dt>Threshold</dt><dd>The score above which a case is called positive; set by the relative costs of false negatives and false positives, \(t^{*} = c_{FP}/(c_{FP}+c_{FN})\) for calibrated probabilities.</dd>
<dt>TRIPOD+AI</dt><dd>The 2024 reporting statement for prediction models, including machine-learning models, in medicine.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Stone M. Cross-validatory choice and assessment of statistical predictions. <em>J R Stat Soc Ser B</em>. 1974;36(2):111–147. <a href="https://doi.org/10.1111/j.2517-6161.1974.tb00994.x">doi:10.1111/j.2517-6161.1974.tb00994.x</a></li>
<li id="r2">Kohavi R. A study of cross-validation and bootstrap for accuracy estimation and model selection. In: <em>Proceedings of the 14th International Joint Conference on Artificial Intelligence (IJCAI-95)</em>. Morgan Kaufmann; 1995:1137–1145. No DOI; proceedings available at <a href="https://www.ijcai.org/Proceedings/95-2/Papers/016.pdf">ijcai.org/Proceedings/95-2/Papers/016.pdf</a></li>
<li id="r3">Varma S, Simon R. Bias in error estimation when using cross-validation for model selection. <em>BMC Bioinformatics</em>. 2006;7:91. <a href="https://doi.org/10.1186/1471-2105-7-91">doi:10.1186/1471-2105-7-91</a></li>
<li id="r4">Hanley JA, McNeil BJ. The meaning and use of the area under a receiver operating characteristic (ROC) curve. <em>Radiology</em>. 1982;143(1):29–36. <a href="https://doi.org/10.1148/radiology.143.1.7063747">doi:10.1148/radiology.143.1.7063747</a></li>
<li id="r5">Fawcett T. An introduction to ROC analysis. <em>Pattern Recognit Lett</em>. 2006;27(8):861–874. <a href="https://doi.org/10.1016/j.patrec.2005.10.010">doi:10.1016/j.patrec.2005.10.010</a></li>
<li id="r6">Saito T, Rehmsmeier M. The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. <em>PLoS ONE</em>. 2015;10(3):e0118432. <a href="https://doi.org/10.1371/journal.pone.0118432">doi:10.1371/journal.pone.0118432</a></li>
<li id="r7">Matthews BW. Comparison of the predicted and observed secondary structure of T4 phage lysozyme. <em>Biochim Biophys Acta</em>. 1975;405(2):442–451. <a href="https://doi.org/10.1016/0005-2795(75)90109-9">doi:10.1016/0005-2795(75)90109-9</a></li>
<li id="r8">Chicco D, Jurman G. The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation. <em>BMC Genomics</em>. 2020;21:6. <a href="https://doi.org/10.1186/s12864-019-6413-7">doi:10.1186/s12864-019-6413-7</a></li>
<li id="r9">Brier GW. Verification of forecasts expressed in terms of probability. <em>Mon Weather Rev</em>. 1950;78(1):1–3. <a href="https://doi.org/10.1175/1520-0493(1950)078<0001:VOFEIT>2.0.CO;2">doi:10.1175/1520-0493(1950)078&lt;0001:VOFEIT&gt;2.0.CO;2</a></li>
<li id="r10">Van Calster B, McLernon DJ, van Smeden M, Wynants L, Steyerberg EW. Calibration: the Achilles heel of predictive analytics. <em>BMC Med</em>. 2019;17:230. <a href="https://doi.org/10.1186/s12916-019-1466-7">doi:10.1186/s12916-019-1466-7</a></li>
<li id="r11">Kaufman S, Rosset S, Perlich C, Stitelman O. Leakage in data mining: formulation, detection, and avoidance. <em>ACM Trans Knowl Discov Data</em>. 2012;6(4):15. <a href="https://doi.org/10.1145/2382577.2382579">doi:10.1145/2382577.2382579</a></li>
<li id="r12">Kapoor S, Narayanan A. Leakage and the reproducibility crisis in machine-learning-based science. <em>Patterns</em>. 2023;4(9):100804. <a href="https://doi.org/10.1016/j.patter.2023.100804">doi:10.1016/j.patter.2023.100804</a></li>
<li id="r13">Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <em>BMJ</em>. 2024;385:e078378. <a href="https://doi.org/10.1136/bmj-2023-078378">doi:10.1136/bmj-2023-078378</a></li>
<li id="r14">Steyerberg EW, Vergouwe Y. Towards better clinical prediction models: seven steps for development and an ABCD for validation. <em>Eur Heart J</em>. 2014;35(29):1925–1931. <a href="https://doi.org/10.1093/eurheartj/ehu207">doi:10.1093/eurheartj/ehu207</a></li>
<li id="r15">Geman S, Bienenstock E, Doursat R. Neural networks and the bias/variance dilemma. <em>Neural Comput</em>. 1992;4(1):1–58. <a href="https://doi.org/10.1162/neco.1992.4.1.1">doi:10.1162/neco.1992.4.1.1</a></li>
<li id="r16">Altman DG, Bland JM. Diagnostic tests 1: sensitivity and specificity. <em>BMJ</em>. 1994;308(6943):1552. <a href="https://doi.org/10.1136/bmj.308.6943.1552">doi:10.1136/bmj.308.6943.1552</a></li>
<li id="r17">Altman DG, Bland JM. Diagnostic tests 2: predictive values. <em>BMJ</em>. 1994;309(6947):102. <a href="https://doi.org/10.1136/bmj.309.6947.102">doi:10.1136/bmj.309.6947.102</a></li>
<li id="r18">Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. Chapter 7, Model assessment and selection. <a href="https://doi.org/10.1007/978-0-387-84858-7">doi:10.1007/978-0-387-84858-7</a></li>
<li id="r19">Raschka S. Model evaluation, model selection, and algorithm selection in machine learning. <em>arXiv</em>. 2018;arXiv:1811.12808. <a href="https://doi.org/10.48550/arXiv.1811.12808">doi:10.48550/arXiv.1811.12808</a></li>
<li id="r20">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r21">Efron B, Hastie T, Johnstone I, Tibshirani R. Least angle regression. <em>Ann Stat</em>. 2004;32(2):407–499. <a href="https://doi.org/10.1214/009053604000000067">doi:10.1214/009053604000000067</a></li>
<li id="r22">Pedregosa F, Varoquaux G, Gramfort A, et al. Scikit-learn: machine learning in Python. <em>J Mach Learn Res</em>. 2011;12:2825–2830. No DOI; <a href="https://jmlr.org/papers/v12/pedregosa11a.html">jmlr.org/papers/v12/pedregosa11a.html</a></li>
<li id="r23">Cawley GC, Talbot NLC. On over-fitting in model selection and subsequent selection bias in performance evaluation. <em>J Mach Learn Res</em>. 2010;11:2079–2107. No DOI; <a href="https://jmlr.org/papers/v11/cawley10a.html">jmlr.org/papers/v11/cawley10a.html</a></li>
<li id="r24">Ambroise C, McLachlan GJ. Selection bias in gene extraction on the basis of microarray gene-expression data. <em>Proc Natl Acad Sci USA</em>. 2002;99(10):6562–6566. <a href="https://doi.org/10.1073/pnas.102102699">doi:10.1073/pnas.102102699</a></li>
<li id="r25">Saeb S, Lonini L, Jayaraman A, Mohr DC, Kording KP. The need to approximate the use-case in clinical machine learning. <em>GigaScience</em>. 2017;6(5):gix019. <a href="https://doi.org/10.1093/gigascience/gix019">doi:10.1093/gigascience/gix019</a></li>
<li id="r26">Steyerberg EW, Vickers AJ, Cook NR, et al. Assessing the performance of prediction models: a framework for traditional and novel measures. <em>Epidemiology</em>. 2010;21(1):128–138. <a href="https://doi.org/10.1097/EDE.0b013e3181c30fb2">doi:10.1097/EDE.0b013e3181c30fb2</a></li>
<li id="r27">Riley RD, Ensor J, Snell KIE, et al. Calculating the sample size required for developing a clinical prediction model. <em>BMJ</em>. 2020;368:m441. <a href="https://doi.org/10.1136/bmj.m441">doi:10.1136/bmj.m441</a></li>
<li id="r28">Ramspek CL, Jager KJ, Dekker FW, Zoccali C, van Diepen M. External validation of prognostic models: what, why, how, when and where? <em>Clin Kidney J</em>. 2021;14(1):49–58. <a href="https://doi.org/10.1093/ckj/sfaa188">doi:10.1093/ckj/sfaa188</a></li>
<li id="r29">Brodersen KH, Ong CS, Stephan KE, Buhmann JM. The balanced accuracy and its posterior distribution. In: <em>Proceedings of the 20th International Conference on Pattern Recognition (ICPR 2010)</em>. IEEE; 2010:3121–3124. <a href="https://doi.org/10.1109/ICPR.2010.764">doi:10.1109/ICPR.2010.764</a></li>
<li id="r30">Davis J, Goadrich M. The relationship between precision-recall and ROC curves. In: <em>Proceedings of the 23rd International Conference on Machine Learning (ICML 2006)</em>. ACM; 2006:233–240. <a href="https://doi.org/10.1145/1143844.1143874">doi:10.1145/1143844.1143874</a></li>
<li id="r31">Vickers AJ, Elkin EB. Decision curve analysis: a novel method for evaluating prediction models. <em>Med Decis Making</em>. 2006;26(6):565–574. <a href="https://doi.org/10.1177/0272989X06295361">doi:10.1177/0272989X06295361</a></li>
<li id="r32">Wong A, Otles E, Donnelly JP, et al. External validation of a widely implemented proprietary sepsis prediction model in hospitalized patients. <em>JAMA Intern Med</em>. 2021;181(8):1065–1070. <a href="https://doi.org/10.1001/jamainternmed.2021.2626">doi:10.1001/jamainternmed.2021.2626</a></li>
<li id="r33">Robin X, Turck N, Hainard A, et al. pROC: an open-source package for R and S+ to analyze and compare ROC curves. <em>BMC Bioinformatics</em>. 2011;12:77. <a href="https://doi.org/10.1186/1471-2105-12-77">doi:10.1186/1471-2105-12-77</a></li>
<li id="r34">Kuhn M. Building predictive models in R using the caret package. <em>J Stat Softw</em>. 2008;28(5):1–26. <a href="https://doi.org/10.18637/jss.v028.i05">doi:10.18637/jss.v028.i05</a></li>
<li id="r35">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. R package. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r36">Harrison D, Rubinfeld DL. Hedonic housing prices and the demand for clean air. <em>J Environ Econ Manag</em>. 1978;5(1):81–102. <a href="https://doi.org/10.1016/0095-0696(78)90006-2">doi:10.1016/0095-0696(78)90006-2</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Chapter 7 is the reference treatment of everything in Sections 1–4; free PDF from the authors' site.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — Chapters 2 and 5 (bias–variance, cross-validation) at an accessible level, with R code; free PDF at statlearning.com.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS). 4th ed. Springer; 2002. — The reference for the R modelling functions used here (<code>lda</code>, <code>glm</code>, <code>knn</code>).</li>
<li>Steyerberg EW. <em>Clinical Prediction Models</em>. 2nd ed. Springer; 2019. — The medical-statistics view of validation, calibration and reporting.</li>
<li>Kuhn M, Johnson K. <em>Applied Predictive Modeling</em>. Springer; 2013. — The <code>caret</code> workflow for the evaluation steps of this supplement.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>pROC package documentation and vignette — <a href="https://xrobin.github.io/pROC/">xrobin.github.io/pROC</a>; ROC/AUC construction and comparison in R.</li>
<li>caret package documentation, "Model training and tuning" — <a href="https://topepo.github.io/caret/">topepo.github.io/caret</a>; the packaged resampling and tuning used across R.</li>
<li>TRIPOD statement website, checklists and explanation documents — <a href="https://www.tripod-statement.org">tripod-statement.org</a>.</li>
<li>Kapoor S, Narayanan A. Leakage model info sheet and checklist — <a href="https://reproducible.cs.princeton.edu">reproducible.cs.princeton.edu</a>.</li>
</ul>
